
# Introduction

[Scrapy](https://scrapy.org/) est un framework permettant de crawler des
sites web et d'en extraire les données de façon structurée.

## Installation

Nous travaillerons dans un environnement
[Anaconda](https://www.anaconda.com/download/), déjà présent sur les
machines de l'ESIEE. Sur vos machines personnelles, télécharger la
distribution correspondant à la version la plus récente de Python.

[Scrapy](https://scrapy.org/) ne fait pas partie de la distribution par
défaut de Python et doit être installé manuellement. Ici, le package est
déjà installé grâce à Pipenv.

Si vous avez besoin d'installer dans un autre cadre.

-   Avec **Pipenv** : `pipenv install scrapy`
-   Avec **Anaconda** : `conda install -c conda-forge scrapy`

Tester la réussite de l'opération dans un interpréteur Python. Avant
installation:

In [1]:
!pip install scrapy

In [1]:
import scrapy

## Architecture

[Scrapy](https://scrapy.org/) est un framework comportant plusieurs
composants.

<img src="images/architecture.png" alt="image" class="align-center" />

L'ensemble du processus est contrôlé par l'**engine** (les termes anglo
saxons ont été retenus pour un meilleur référencement dans la
[documentation officielle](https://docs.scrapy.org/en/latest/)).

Le framework est articulé avec plusieurs composants qui gèrent chacun un
rôle différent. Nous allons les détailler.

-   Les **Spiders** : permettent de naviguer sur un site et de
    référencer les règles d'extraction de la donnée.
-   Les **Pipelines** : font le lien entre la donnée brute et des objets
    structurés
-   Les **Middlewares** : permettent d'effectuer des transformations sur
    les objets ou sur les requêtes exécutées par l'engine.
-   Le **Scheduler** : gère l'ordre et le timing des requêtes
    effectuées.

## Fonctionnement

[Scrapy](https://scrapy.org/) est entièrement organisé autour d'un
composant central : l'*engine*.

Le rôle de l'*engine* est de contrôler le flux de données entre les
différents composants du système.

1.  En particulier, il est chargé de récupérer les *requests* définies
    dans les *spiders*
2.  Ces *requests* sont ensuite fournies au *scheduler* qui se charge de
    leur ordonnancement
3.  Les *requests* sont présentées selon cet ordonnancement à
    l'*engine*...
4.  ... qui les transmet au *downloader*
5.  Le *downloader* effectue la *request* et transmet la *response* (le
    contenu de la page web) à l'*engine*...
6.  ... puis l'envoie au *spider* pour traitement
7.  Le *spider* génére des *items* qui sont transmis à l'*engine*
8.  Les *items* sont ensuite poussés dans un pipeline pour nettoyage,
    validation et stockage

Ce processus est répété jusqu'à épuisement des requêtes.

[Scrapy](https://scrapy.org/) est un [framework orienté
événements](https://en.wikipedia.org/wiki/Event-driven_architecture)
(basé sur [Twisted](https://twistedmatrix.com/)) permettant une
programmation asynchrone (non bloquante). C'est particulièrement
intéressant dans les opérations de scraping, puisque **le programme
n'attend pas le résultat d'une requête pour en lancer une autre**.

En effet, lorsque l'on sollicite une ressource (requête réseau, système
de fichier, etc.) en mode bloquant, l'exécution du programme est
suspendue le temps que la transaction avec la ressource se termine (par
exemple le temps qu'une page web soit complètement téléchargée).
L'intérêt de faire des appels non bloquants, c'est que l'on peut gérer
de multiples téléchargements en parallèle, et que le programme peut
continuer à tourner pendant ce temps.

# Un scraping élémentaire

Avant de rentrer dans les détails du framework, nous allons mettre en
oeuvre un premier script permettant de récupérer l'information présente
sur [la page web](http://evene.lefigaro.fr/citations/winston-churchill)
recensant les citations de [Sir Winston
Churchill](https://en.wikipedia.org/wiki/Winston_Churchill).

**Exercice**

Examiner le code source de cette page avec l'inspecteur de votre
navigateur. Identifier les éléments contenant l'information recherchée,
ici la chaîne de caractères contenant la citation proprement dite.

## Le code source

Le code utilisé est le suivant:

In [2]:
# %load citations_churchill_spider1.py
import scrapy

class ChurchillQuotesSpider(scrapy.Spider):
    name = "citations de Churchill"
    start_urls = ["http://evene.lefigaro.fr/citations/winston-churchill",]

    def parse(self, response):
        for cit in response.xpath('//div[@class="figsco__quote__text"]'):
            text_value = cit.xpath('a/text()').extract_first()
            yield { 'text' : text_value }

## Le fonctionnement

Le fonctionnement est le suivant:

-   On importe le module [Scrapy](https://scrapy.org/) (3)
-   et on définit une sous classe de `scrapy.Spider` (5)
-   la variable `start_urls` contient la liste des pages à scraper (7)
-   On redéfinit la méthode $parse$ dont la signature est définie dans
    la classe mère (9)
-   L'objet
    [response](https://docs.scrapy.org/en/latest/topics/request-response.html#response-objects)
    représente la réponse à la requête HTTP (l'attribut $text$ permet
    d'accéder à son contenu). On recherche ensuite tous les containers
    `<div>` identifiés dans l'exercice précédent. Ici la page est
    particulièrement bien structurée et les citations disposent de leur
    propre container, identifié par l'attribut `class` de valeur
    `figsco__quote__text`. La sélection se fait par une expression
    [XPath](https://en.wikipedia.org/wiki/XPath), un langage de
    sélection de noeud dans un document XML (10). En langage naturel, la
    requête pourrait se formuler : "On recherche tous les containers
    `<div>` dont la valeur de l'attribut `class` est égal à
    `figsco__quote__text`".
-   Pour chaque résultat, on construit un dictionnaire dont la clé est
    `text` et la valeur le contenu du lien `<a>`. Ce résultat est fourni
    par un générateur ($yield$) (12).

On lance le scraping depuis un terminal:

In [16]:
!scrapy runspider citations_churchill_spider1.py

2019-12-11 09:26:44 [scrapy.utils.log] INFO: Scrapy 1.5.1 started (bot: scrapybot)
2019-12-11 09:26:44 [scrapy.utils.log] INFO: Versions: lxml 4.3.2.0, libxml2 2.9.4, cssselect 1.0.3, parsel 1.5.0, w3lib 1.20.0, Twisted 18.9.0, Python 3.7.3 (default, Apr  3 2019, 05:39:12) - [GCC 8.3.0], pyOpenSSL 19.0.0 (OpenSSL 1.1.1c  28 May 2019), cryptography 2.6.1, Platform Linux-4.19.0-6-amd64-x86_64-with-debian-10.1
2019-12-11 09:26:44 [scrapy.crawler] INFO: Overridden settings: {'SPIDER_LOADER_WARN_ONLY': True}
2019-12-11 09:26:44 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.memusage.MemoryUsage',
 'scrapy.extensions.logstats.LogStats']
2019-12-11 09:26:44 [scrapy.middleware] INFO: Enabled downloader middlewares:
['scrapy.downloadermiddlewares.httpauth.HttpAuthMiddleware',
 'scrapy.downloadermiddlewares.downloadtimeout.DownloadTimeoutMiddleware',
 'scrapy.downloadermiddlewares.defaultheade

On y trouve des informations sur les paramètres
utilisés:

In [5]:
2018-01-10 17:21:05 [scrapy.utils.log] INFO: Scrapy 1.4.0 started (bot: scrapybot)
2018-01-10 17:21:05 [scrapy.utils.log] INFO: Overridden settings: {'SPIDER_LOADER_WARN_ONLY': True}

SyntaxError: invalid token (<ipython-input-5-e17ba2a0ae1a>, line 1)

les
[extensions](https://docs.scrapy.org/en/latest/topics/extensions.html)
...:

In [ ]:
2018-01-10 17:21:05 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
'scrapy.extensions.telnet.TelnetConsole',
'scrapy.extensions.logstats.LogStats']

[Les composants middleware
downloader](https://docs.scrapy.org/en/latest/topics/downloader-middleware.html)
... :

In [ ]:
2018-01-10 17:21:05 [scrapy.middleware] INFO: Enabled downloader middlewares:
['scrapy.downloadermiddlewares.httpauth.HttpAuthMiddleware',
'scrapy.downloadermiddlewares.downloadtimeout.DownloadTimeoutMiddleware',
'scrapy.downloadermiddlewares.defaultheaders.DefaultHeadersMiddleware',
'scrapy.downloadermiddlewares.useragent.UserAgentMiddleware',
'scrapy.downloadermiddlewares.retry.RetryMiddleware',
'scrapy.downloadermiddlewares.redirect.MetaRefreshMiddleware',
'scrapy.downloadermiddlewares.httpcompression.HttpCompressionMiddleware',
'scrapy.downloadermiddlewares.redirect.RedirectMiddleware',
'scrapy.downloadermiddlewares.cookies.CookiesMiddleware',
'scrapy.downloadermiddlewares.httpproxy.HttpProxyMiddleware',
'scrapy.downloadermiddlewares.stats.DownloaderStats']

Idem pour [les composants middleware
spider](https://docs.scrapy.org/en/latest/topics/spider-middleware.html)
...:

In [ ]:
2018-01-10 17:21:05 [scrapy.middleware] INFO: Enabled spider middlewares:
['scrapy.spidermiddlewares.httperror.HttpErrorMiddleware',
'scrapy.spidermiddlewares.offsite.OffsiteMiddleware',
'scrapy.spidermiddlewares.referer.RefererMiddleware',
'scrapy.spidermiddlewares.urllength.UrlLengthMiddleware',
'scrapy.spidermiddlewares.depth.DepthMiddleware']

Aucun
[pipeline](https://docs.scrapy.org/en/latest/topics/item-pipeline.html)
n'est activé :

In [ ]:
2018-01-10 17:21:05 [scrapy.middleware] INFO: Enabled item pipelines:
[]

**Exercice**

Identifier la position des [composants middleware
downloader](https://docs.scrapy.org/en/latest/topics/downloader-middleware.html),
des [composants middleware
spider](https://docs.scrapy.org/en/latest/topics/spider-middleware.html)
et du
[pipeline](https://docs.scrapy.org/en/latest/topics/item-pipeline.html)
dans $l'architecture <Introduction>$

L'exécution du scraping proprement dit débute :

In [ ]:
2018-01-10 17:21:05 [scrapy.core.engine] INFO: Spider opened
2018-01-10 17:21:05 [scrapy.extensions.logstats] INFO: Crawled 0 pages (at 0 pages/min), scraped 0 items (at 0 items/min)
2018-01-10 17:21:05 [scrapy.extensions.telnet] DEBUG: Telnet console listening on 127.0.0.1:6023

La première URL est poussée par le scheduler:

In [ ]:
2018-01-10 17:21:05 [scrapy.core.engine] DEBUG: Crawled (200) <GET http://evene.lefigaro.fr/citations/winston-churchill> (referer: None)

## Les résultats

Les résultats sont fournis par le générateur défini dans la méthode
$parse$ dans un dictionnaire. Ils contiennent le texte des citations
dans la valeur de la clé `text` :

In [ ]:
2018-01-10 17:21:05 [scrapy.core.scraper] DEBUG: Scraped from <200 http://evene.lefigaro.fr/citations/winston-churchill>
{'text': '“Le vice inhérent au capitalisme consiste en une répartition inégale des richesses. La vertu inhérente au socialisme consiste en une égale répartition de la misère.”'}
2018-01-10 17:21:05 [scrapy.core.scraper] DEBUG: Scraped from <200 http://evene.lefigaro.fr/citations/winston-churchill>
{'text': "Faire le bien, éviter le mal, c'est ça le paradis."}

## Les statistiques

Une fois le scraping effectué, quelques statistiques sont affichées sur
le terminal:

In [ ]:
2018-01-10 17:21:05 [scrapy.core.engine] INFO: Closing spider (finished)
2018-01-10 17:21:05 [scrapy.statscollectors] INFO: Dumping Scrapy stats:
{'downloader/request_bytes': 242,
'downloader/request_count': 1,
'downloader/request_method_count/GET': 1,
'downloader/response_bytes': 17435,
'downloader/response_count': 1,
'downloader/response_status_count/200': 1,
'finish_reason': 'finished',
'finish_time': datetime.datetime(2018, 1, 10, 16, 21, 5, 858347),
'item_scraped_count': 16,
'log_count/DEBUG': 18,
'log_count/INFO': 7,
'response_received_count': 1,
'scheduler/dequeued': 1,
'scheduler/dequeued/memory': 1,
'scheduler/enqueued': 1,
'scheduler/enqueued/memory': 1,
'start_time': datetime.datetime(2018, 1, 10, 16, 21, 5, 645347)}
2018-01-10 17:21:05 [scrapy.core.engine] INFO: Spider closed (finished)

On observe notamment que notre code permet de récupérer la taille de la
page web (17435 bytes), le temps d'exécution à partir des valeurs
`finish_time` et `start_time`, le nombre d'items scrapés (16), etc...

**Exercice**

Les citations extraites sont elles toutes de [Sir Winston
Churchill](https://en.wikipedia.org/wiki/Winston_Churchill) ? Il sera
peut être nécessaire de modifier le sélecteur XPath. Nous verrons ça
lorsque il faudra récupérer les données relative à l'auteur.

## Modifier les données

Il est parfois nécessaire de faire un traitement sur les données
scrapées, pour ajouter ou retirer de l'information.

**Exercice**

Retirer les caractères `“` et `”` qui délimitent la citation. Ces
caractères sont identifiés en Unicode comme [LEFT DOUBLE QUOTATION
MARK](http://www.fileformat.info/info/unicode/char/201c/index.htm) et
[RIGHT DOUBLE QUOTATION
MARK](http://www.fileformat.info/info/unicode/char/201d/index.htm).

## Plus de données

Il est souvent nécessaire de récupérer plusieurs informations relatives
à un même item. Dans cet exemple, il est judicieux d'associer à la
citation le nom de son auteur, en allant chercher cette information au
plus près du texte lui même.

**Exercice**

Examiner le code source de la page web et identifier la structuration de
la donnée associée à l'auteur. En déduire l'expression XPath permettant
de la récupérer. S'assurer que seules les citations de [Sir Winston
Churchill](https://en.wikipedia.org/wiki/Winston_Churchill) sont
extraites. Ajouter une clé `author` au dictionnaire retourné par le
$yield$ dont la valeur est précisément la chaîne de caractères contenant
l'auteur.

Un exemple de dictionnaire retourné:

In [ ]:
{   'text': "“Si deux hommes ont toujours la même opinion, l'un d'eux est de trop.”", 
    'author': 'Winston Churchill'}

Récupération des citations avec nom de l'auteur, sans les caractères `“` et `”`

In [ ]:
# %load citations_churchill_spider2.py
import scrapy

class ChurchillQuotesSpider(scrapy.Spider):
    name = "citations de Churchill"
    start_urls = ["http://evene.lefigaro.fr/citations/winston-churchill",]

    def parse(self, response):
        #réponse à la requête HTTP (l'attribut $text$ permet
        #d'accéder à son contenu)
        for cit in response.xpath('//li[@class="figsco__selection__list__evene__list__item"]'):
            text_container = cit.xpath('//div[@class="figsco__quote__text"]')
            author_container = cit.xpath('//div[@class="figsco__fake__col-9"]')
            
            text_value = text_container.xpath('a/text()').extract_first().translate({ord(i): None for i in 'u"\u201C"u"\u201D"'})
            author_name = author_container.xpath('a/text()').extract_first()
            
            yield { 'text' : text_value,
                    'author' : author_name }




In [44]:
!scrapy runspider citations_churchill_spider2.py

2019-12-11 09:51:46 [scrapy.utils.log] INFO: Scrapy 1.5.1 started (bot: scrapybot)
2019-12-11 09:51:46 [scrapy.utils.log] INFO: Versions: lxml 4.3.2.0, libxml2 2.9.4, cssselect 1.0.3, parsel 1.5.0, w3lib 1.20.0, Twisted 18.9.0, Python 3.7.3 (default, Apr  3 2019, 05:39:12) - [GCC 8.3.0], pyOpenSSL 19.0.0 (OpenSSL 1.1.1c  28 May 2019), cryptography 2.6.1, Platform Linux-4.19.0-6-amd64-x86_64-with-debian-10.1
2019-12-11 09:51:46 [scrapy.crawler] INFO: Overridden settings: {'SPIDER_LOADER_WARN_ONLY': True}
2019-12-11 09:51:46 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.memusage.MemoryUsage',
 'scrapy.extensions.logstats.LogStats']
2019-12-11 09:51:46 [scrapy.middleware] INFO: Enabled downloader middlewares:
['scrapy.downloadermiddlewares.httpauth.HttpAuthMiddleware',
 'scrapy.downloadermiddlewares.downloadtimeout.DownloadTimeoutMiddleware',
 'scrapy.downloadermiddlewares.defaultheade

Pour lancer l'exécution de la spider :

> \$ scrapy runspider spiders/citations\_churchill\_spider2.py

On peut aussi vouloir stocker les données extraites :

> \$ scrapy runspider spiders/citations\_churchill\_spider2.py -o
> data/citation.json -t json


# Votre premier projet


Dans un premier temps vous devez créer un projet Scrapy avec la commande
:

In [46]:
!scrapy startproject newscrawler

Error: scrapy.cfg already exists in /user/lamyvert/homedir/DSIA-4201C/DataEngineerTools/2Scrapy/newscrawler


Cette commande va créer un dossier `monprojet` contenant les éléments
suivants correspondant au squelette :

In [ ]:
newscrawler/
    scrapy.cfg            # Options de déploiement

    newscrawler/             # Le module Python contenant les informations
        __init__.py

        items.py          # Fichier contenant les items

        middlewares.py    # Fichier contenant les middlewares

        pipelines.py      # Fichier contenant les pipelines

        settings.py       # Fichier contenant les paramètres du projet

        spiders/          # Dossier contenant toutes les spiders
            __init__.py

# Votre première Spider

Une Spider est une classe Scrapy qui permet de mettre en place toute
l'architecture complexe vue dans l'introduction. Pour définir une
spider, il vous faut hériter de la classe $scrapy.Spider$. La seule
chose à faire est de définir la première requête à effectuer et comment
suivre les liens. La Spider s'arrêtera lorsqu'elle aura parcouru tous
les liens qu'on lui a demandé de suivre.

Pour créer une Spider on utilise la syntaxe:

In [47]:
!scrapy genspider <SPIDER_NAME> <DOMAIN_NAME>

/bin/sh: 1: Syntax error: redirection unexpected


Par exemple,

In [3]:
!cd newscrawler && scrapy genspider lemonde lemonde.fr

Spider 'lemonde' already exists in module:
  newscrawler.spiders.lemonde


Cette commande permet de créer une spider appelée `lemonde` pour scraper
le domaine `lemonde.fr`. Cela crée le fichier Python
`spiders/lemonde.py` suivant :

In [4]:
# %load newscrawler/newscrawler/spiders/lemonde.py
import scrapy


class LemondeSpider(scrapy.Spider):
    name = 'lemonde'
    allowed_domains = ['lemonde.fr']
    start_urls = ['http://lemonde.fr/']

    def parse(self, response):
        pass


Une bonne pratique pour commencer à développer une Spider est de passer
par l'interface Shell proposée par Scrapy. Elle permet de récupérer un
objet `Response` et de tester les méthodes de récupération des données.

# ATTENTION : Les commandes scrapy shell doivent être lancées dans un terminal 

In [ ]:
scrapy shell 'http://lemonde.fr'

Pour les utilisateurs de windows il vous faut mettre des doubles quotes
:

In [ ]:
scrapy shell "http://lemonde.fr"

Scrapy lance un kernel Python

In [ ]:
2018-12-02 16:05:50 [scrapy.utils.log] INFO: Scrapy 1.3.3 started (bot: newscrawler)
2018-12-02 16:05:50 [scrapy.utils.log] INFO: Overridden settings: {'BOT_NAME': 'newscrawler', 'DUPEFILTER_CLASS': 'scrapy.dupefilters.BaseDupeFilter', 'LOGSTATS_INTERVAL': 0, 'NEWSPIDER_MODULE': 'newscrawler.spiders', 'ROBOTSTXT_OBEY': True, 'SPIDER_MODULES': ['newscrawler.spiders']}
2018-12-02 16:05:50 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
'scrapy.extensions.telnet.TelnetConsole']
2018-12-02 16:05:50 [scrapy.middleware] INFO: Enabled downloader middlewares:
['scrapy.downloadermiddlewares.robotstxt.RobotsTxtMiddleware',
'scrapy.downloadermiddlewares.httpauth.HttpAuthMiddleware',
'scrapy.downloadermiddlewares.downloadtimeout.DownloadTimeoutMiddleware',
'scrapy.downloadermiddlewares.defaultheaders.DefaultHeadersMiddleware',
'scrapy.downloadermiddlewares.useragent.UserAgentMiddleware',
'scrapy.downloadermiddlewares.retry.RetryMiddleware',
'scrapy.downloadermiddlewares.redirect.MetaRefreshMiddleware',
'scrapy.downloadermiddlewares.httpcompression.HttpCompressionMiddleware',
'scrapy.downloadermiddlewares.redirect.RedirectMiddleware',
'scrapy.downloadermiddlewares.cookies.CookiesMiddleware',
'scrapy.downloadermiddlewares.stats.DownloaderStats']
2018-12-02 16:05:50 [scrapy.middleware] INFO: Enabled spider middlewares:
['scrapy.spidermiddlewares.httperror.HttpErrorMiddleware',
'scrapy.spidermiddlewares.offsite.OffsiteMiddleware',
'scrapy.spidermiddlewares.referer.RefererMiddleware',
'scrapy.spidermiddlewares.urllength.UrlLengthMiddleware',
'scrapy.spidermiddlewares.depth.DepthMiddleware']
2018-12-02 16:05:50 [scrapy.middleware] INFO: Enabled item pipelines:
[]
2018-12-02 16:05:50 [scrapy.extensions.telnet] DEBUG: Telnet console listening on 127.0.0.1:6023
2018-12-02 16:05:50 [scrapy.core.engine] INFO: Spider opened
2018-12-02 16:05:50 [scrapy.core.engine] DEBUG: Crawled (200) <GET https://www.lemonde.fr/robots.txt> (referer: None)
2018-12-02 16:05:50 [scrapy.core.engine] DEBUG: Crawled (200) <GET https://www.lemonde.fr/> (referer: None)
2018-12-02 16:05:54 [traitlets] DEBUG: Using default logger
2018-12-02 16:05:54 [traitlets] DEBUG: Using default logger
[s] Available Scrapy objects:
[s]   scrapy     scrapy module (contains scrapy.Request, scrapy.Selector, etc)
[s]   crawler    <scrapy.crawler.Crawler object at 0x10fc38c18>
[s]   item       {}
[s]   request    <GET https://www.lemonde.fr/>
[s]   response   <200 https://www.lemonde.fr/>
[s]   settings   <scrapy.settings.Settings object at 0x113bb0898>
[s]   spider     <DefaultSpider 'default' at 0x113e60cc0>
[s] Useful shortcuts:
[s]   fetch(url[, redirect=True]) Fetch URL and update local objects (by default, redirects are followed)
[s]   fetch(req)                  Fetch a scrapy.Request and update local objects
[s]   shelp()           Shell help (print this help)
[s]   view(response)    View response in a browser

Grâce à cette interface, vous avez accès à plusieurs objets comme la
`Response`, la `Request`, la `Spider` par exemple. Vous pouvez aussi
exécuter `view(response)` pour afficher ce que Scrapy récupère dans un
navigateur.

In [ ]:
In [1]: response
Out[1]: <200 https://www.lemonde.fr/>

In [3]: request
Out[3]: <GET https://www.lemonde.fr/>

In [4]: type(request)
Out[4]: scrapy.http.request.Request

In [5]: spider
Out[5]: <LemondeSpider 'lemonde' at 0x1080fccc0>

In [6]: type(spider)
Out[6]: monprojet.spiders.lemonde.LeMondeSpider

Ici on voit que la Spider est une instance de LemondeSpider. Lorsqu'on
lance le $scrapy shell$ scrapy va chercher dans les spiders si il
correspond au lien passé en paramètre, si oui , il l'utilise sinon une
$DefaultSpider$ est instanciée.

## Vos premières requêtes

On peut commencer à regarder comment extraire les données de la page web
en utilisant le langage de requêtes proposé par Scrapy. Il existe deux
types de requêtes : les requêtes `css` et `xpath`. Les requêtes `xpath`
sont plus complexes mais plus puissantes que les requêtes `css`. Dans le
cadre de ce tutorial, nous allons uniquement aborder les requêtes `css`,
elles nous suffiront pour extraire les données dont nous avons besoin
(en interne, Scrapy transforme les requêtes `css`en requêtes `xpath`.

Que ce soit les requêtes `css` ou `xpath`, elles crééent des sélecteurs
de différents types. Quelques exemples :

Pour récupérer le titre d'une page :

In [ ]:
In [1]: response.css('title')
Out[1]: [<Selector xpath='descendant-or-self::title' data='<title>Le Monde.fr - Actualités et Infos'>]

On récupère une liste de sélecteurs correspondant à la requête `css`
appelée. La requête précédente était unique, d'autres requêtes moins
restrictives permettent de récupérer plusieurs résultats. Par exemple
pour rechercher l'ensemble des liens présents sur la page, on va
rechercher les balises HTML `<a></a>`

In [ ]:
In [5]: response.css("a")[0:10]
Out[5]:
[<Selector xpath='descendant-or-self::a' data='<a target="_blank" data-target="jelec-he'>,
<Selector xpath='descendant-or-self::a' data='<a href="/"> <div class="logo__lemonde l'>,
<Selector xpath='descendant-or-self::a' data='<a href="https://secure.lemonde.fr/sfuse'>,
<Selector xpath='descendant-or-self::a' data='<a href="https://abo.lemonde.fr/#xtor=CS'>,
<Selector xpath='descendant-or-self::a' data='<a href="/" class="Burger__right-arrow j'>,
<Selector xpath='descendant-or-self::a' data='<a href="/" class="Burger__right-arrow j'>,
<Selector xpath='descendant-or-self::a' data='<a href="#" class="js-dropdown Burger__r'>,
<Selector xpath='descendant-or-self::a' data='<a href="/mouvement-des-gilets-jaunes/" '>,
<Selector xpath='descendant-or-self::a' data='<a href="/carlos-ghosn/" data-suggestion'>,
<Selector xpath='descendant-or-self::a' data='<a href="/implant-files/" data-suggestio'>]

Pour récupérer le texte contenu dans les balises, on passe le paramètre
`<TAG>::text`. Par exemple :

In [6]: response.css("title::text")
Out[6]: [<Selector xpath='descendant-or-self::title/text()' data='Le Monde.fr - Actualités et Infos en Fra'>]

### Exercice

  Comparer les résultats des deux requêtes `response.css('title')` et
`response.css('title::text')`.

In [50]:
In [4]: response.css('title::text')
Out[4]: [<Selector xpath='descendant-or-self::title/text()' data='Le Monde.fr - Actualités et Infos en Fra'>]

In [5]: response.css('title')
Out[5]: [<Selector xpath='descendant-or-self::title' data='<title>Le Monde.fr - Actualités et Infos'>]


SyntaxError: invalid syntax (<ipython-input-50-7f659a1d09b2>, line 2)

Maintenant pour extraire les données des selecteurs on utilise l'une des
deux méthodes suivantes : - `extract()` permet de récupérer une liste
des données extraites de tous les sélecteurs - `extract_first()` permet
de récupérer une `String` provenant du premier sélecteur de la liste.

In [ ]:
In [7]: response.css('title::text').extract_first()
Out[7]: 'Le Monde.fr - Actualités et Infos en France et dans le monde'

On peut récupérer un attribut d'une balise avec la syntaxe
`<TAG>::attr(<ATTRIBUTE_NAME>)` :

Par exemple, les liens sont contenus dans un attribut `href`.

In [ ]:
In [9]: response.css('a::attr(href)')[0:10]
Out[9]:
[<Selector xpath='descendant-or-self::a/@href' data='https://journal.lemonde.fr'>,
<Selector xpath='descendant-or-self::a/@href' data='/'>,
<Selector xpath='descendant-or-self::a/@href' data='https://secure.lemonde.fr/sfuser/connexi'>,
<Selector xpath='descendant-or-self::a/@href' data='https://abo.lemonde.fr/#xtor=CS1-454[CTA'>,
<Selector xpath='descendant-or-self::a/@href' data='/'>,
<Selector xpath='descendant-or-self::a/@href' data='/'>,
<Selector xpath='descendant-or-self::a/@href' data='#'>,
<Selector xpath='descendant-or-self::a/@href' data='/mouvement-des-gilets-jaunes/'>,
<Selector xpath='descendant-or-self::a/@href' data='/carlos-ghosn/'>,
<Selector xpath='descendant-or-self::a/@href' data='/implant-files/'>]

Comme vu précédemment, si on veut récupérer la liste des liens de la page on applique la méthode $extract()$

In [ ]:
In [11]: response.css('a::attr(href)').extract()[0:10]
Out[11]:
['https://journal.lemonde.fr',
'/',
'https://secure.lemonde.fr/sfuser/connexion',
'https://abo.lemonde.fr/#xtor=CS1-454[CTA_LMFR]-[HEADER]-5-[Home]',
'/',
'/',
'#',
'/mouvement-des-gilets-jaunes/',
'/carlos-ghosn/',
'/implant-files/']

Les liens dans une page HTML sont souvent codés de manière relative par
rapport à la page courante. La méthode de l'objet `Response` peut être
utilisée pour recréer l'url complet.

Un exemple sur le 4e élément :

In [ ]:
In [14]: response.urljoin(response.css('a::attr(href)').extract()[8])
Out[14]: 'https://www.lemonde.fr/carlos-ghosn/'

alors que

In [ ]:
In [15]: response.css('a::attr(href)').extract()[8]
Out[15]: '/carlos-ghosn/'

### Exercice : 

Utiliser une liste compréhension pour transformer les 10
premiers liens relatifs récupérés par la méthode `extract()` en liens
absolus.    

Le résultat doit ressembler à :

In [ ]:
Out[23]: 
['https://journal.lemonde.fr',
'https://www.lemonde.fr/',
'https://secure.lemonde.fr/sfuser/connexion',
'https://abo.lemonde.fr/#xtor=CS1-454[CTA_LMFR]-[HEADER]-5-[Home]',
'https://www.lemonde.fr/',
'https://www.lemonde.fr/',
'https://www.lemonde.fr/',
'https://www.lemonde.fr/mouvement-des-gilets-jaunes/',
'https://www.lemonde.fr/carlos-ghosn/',
'https://www.lemonde.fr/implant-files/']

In [ ]:
In [11]: [response.urljoin(link) for link in response.css('a::attr(href)').extract()[:10]]
Out[11]: 
['https://journal.lemonde.fr',
 'https://www.lemonde.fr/',
 'https://secure.lemonde.fr/sfuser/connexion',
 'https://abo.lemonde.fr/#xtor=CS1-454[CTA_LMFR]-[HEADER]-5-[Home]',
 'https://www.lemonde.fr/',
 'https://www.lemonde.fr/',
 'https://www.lemonde.fr/',
 'https://www.lemonde.fr/mobilisation-du-5-decembre/',
 'https://www.lemonde.fr/debat-sur-les-retraites/',
 'https://www.lemonde.fr/cop25/']


## Des requêtes plus complexes

On peut créer des requêtes plus complexes en utilisant à la fois la
structuration HTML du document mais également la couche de présentation
CSS. On utilise l'inspecteur de `Google Chrome` pour identifier le type
et l'identifiant de la balise contenant les informations.

Il y a au moins deux choses à savoir en `css` :  

-   Les `.` représentent les classes
-   Les `#` représentent les id

On se propose de récupérer toutes les sous-catégories de news dans la
catégorie **Actualités**. On remarque en utilisant l'inspecteur
d'élement de Chrome que toutes les catégories sont rangées dans une
balise avec l'id $#nav-markup$ ensuite dans les classes $Nav__item$.

A partir de cette structure HTML on peut construire la requête suivante
pour récupérer la barre de navigation:

In [ ]:
In [19]: response.css("#nav-markup")
Out[19]: [<Selector xpath="descendant-or-self::*[@id = 'nav-markup']" data='<ul id="nav-markup"> <li class="Nav__ite'>]

Ensuite pour récupérer les différentes catégories :

In [ ]:
In [24]: response.css("#nav-markup .Nav__item")
Out[24]:
[<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item js-burger-to-show N'>,
<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item Nav__item--home Nav'>,
<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item"> <a href="/" class'>,
<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item"> <a href="#" class'>,
<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item"> <a href="#" class'>,
<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item"> <a href="#" class'>,
<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item"> <a href="#" class'>,
<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item"> <a href="#" class'>,
<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item"> <a href="#" class'>,
<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item"> <a href="#" class'>,
<Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item"> <a href="/recherc'>]

On veut maintenant retourner tous les liens présents dans cette
catégorie. On remarque qu'elle apparait à la 4eme position.

In [ ]:
In [34]: response.css("#nav-markup .Nav__item")[3]
Out[34]: <Selector xpath="descendant-or-self::*[@id = 'nav-markup']/descendant-or-self::*/*[@class and contains(concat(' ', normalize-space(@class), ' '), ' Nav__item ')]" data='<li class="Nav__item"> <a href="#" class'>

Maintenant pour récupérer tous les liens on peut chainer les requêtes.
On accède alors à toutes les balises $a$.

In [ ]:
In [35]: response.css("#nav-markup .Nav__item")[3].css("a")
Out[35]:
[<Selector xpath='descendant-or-self::a' data='<a href="#" class="js-dropdown Burger__r'>,
<Selector xpath='descendant-or-self::a' data='<a href="/mouvement-des-gilets-jaunes/" '>,
<Selector xpath='descendant-or-self::a' data='<a href="/carlos-ghosn/" data-suggestion'>,
<Selector xpath='descendant-or-self::a' data='<a href="/implant-files/" data-suggestio'>,
<Selector xpath='descendant-or-self::a' data='<a href="/climat/" data-suggestion>Clima'>,
<Selector xpath='descendant-or-self::a' data='<a href="/affaire-khashoggi/" data-sugge'>,
<Selector xpath='descendant-or-self::a' data='<a href="/emmanuel-macron/" data-suggest'>,
<Selector xpath='descendant-or-self::a' data='<a href="/ukraine/" data-suggestion>Ukra'>,
<Selector xpath='descendant-or-self::a' data='<a href="/russie/" data-suggestion>Russi'>,
<Selector xpath='descendant-or-self::a' data='<a href="/referendum-sur-le-brexit/" dat'>,
<Selector xpath='descendant-or-self::a' data='<a href="/harcelement-sexuel/" data-sugg'>,
<Selector xpath='descendant-or-self::a' data='<a href="/actualite-en-continu/" data-su'>,
<Selector xpath='descendant-or-self::a' data='<a href="/international/">International<'>,
<Selector xpath='descendant-or-self::a' data='<a href="/politique/">Politique</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/societe/">Société</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/les-decodeurs/">Les Décodeurs<'>,
<Selector xpath='descendant-or-self::a' data='<a href="/sport/">Sport</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/planete/">Planète</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/sciences/">Sciences</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/campus/">M Campus</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/afrique/">Le Monde Afrique</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/pixels/">Pixels</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/actualite-medias/">Médias</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/sante/">Santé</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/big-browser/">Big Browser</a>'>,
<Selector xpath='descendant-or-self::a' data='<a href="/disparitions/">Disparitions</a'>]

Et pour récupérer les titres :

In [ ]:
In [37]: response.css("#nav-markup .Nav__item")[3].css("a::text").extract()
Out[37]:
['Actualités',
'Mouvement des "gilets jaunes"',
'Carlos Ghosn',
'Implant Files',
'Climat',
'Affaire Khashoggi',
'Emmanuel Macron',
'Ukraine',
'Russie',
'Brexit',
'Harcèlement sexuel',
'Toute l’actualité en continu',
'International',
'Politique',
'Société',
'Les Décodeurs',
'Sport',
'Planète',
'Sciences',
'M Campus',
'Le Monde Afrique',
'Pixels',
'Médias',
'Santé',
'Big Browser',
'Disparitions']

Le shell Scrapy permet de définir la structure des requêtes et de
s'assurer de la pertinence du résultat retourné. Pour automatiser le
processus, il faut intégrer cette syntaxe au code Python des modules de
spider définis dans la structure du projet.

## Intégration des requêtes

Le squelette de la classe `LeMondeSpider` généré lors de la création du
projet doit maintenant être enrichi. Par défaut 3 attributs et une
méthode `parse()` ont été créés :

-   `name` permet d'identifier sans ambiguïté la spider dans le code.
-   `allowed_domain` permet de filtrer les requêtes et forcer la spider
    à rester sur une liste de domaines.
-   `starts_urls` est la liste des urls d'où la spider va partir pour
    commencer son scraping.
-   `parse()` est une méthode héritée de la classe `scrapy.Spider`. Elle
    doit être redéfinie selon les requêtes que l'on doit effectuer et
    sera appelée sur l'ensemble des urls contenus dans la liste
    `starts_urls`.

`parse()` est une fonction `callback` qui sera appelée automatiquement
sur chaque objet `Response` retourné par la requête. Cette fonction est
appelée de manière asynchrone. Plusieurs requêtes peuvent ainsi être
lancées en parallèles sans bloquer le thread principal. L'objet
`Response` passé en paramètre est le même que celui mis à disposition
lors de l'exécution du Scrapy Shell.

In [5]:
def parse(self, response):
    title = response.css('title::text').extract_first()
    all_links = {
        name:response.urljoin(url) for name, url in zip(
        response.css("#nav-markup .Nav__item")[3].css("a::text").extract(),
        response.css("#nav-markup .Nav__item")[3].css("a::attr(href)").extract())
    }
    yield {
        "title":title,
        "all_links":all_links
    }

La fonction est un générateur (`yield`) et retourne un dictionnaire
composé de deux éléments :

-   Le titre de la page;
-   La liste des liens sortants sous forme de String.

Pour le moment cette spider ne parcourt que la page d'accueil, ce qui
n'est pas très productif.

## Votre premier scraper

Récupérer les données sur un ensemble de pages webs nécessite d'explorer
en profondeur la structure du site en suivant tout ou partie des liens
rencontrés.

La spider peut se `balader` sur un site assez efficacement. Il suffit de
lui indiquer comment faire. Il faut spécifier à Scrapy de générer une
requête vers une nouvelle page en construisant l'objet `Request`
correspondant. Ce nouvel objet `Request` est alors inséré dans le
scheduler de Scrapy. On peut évidemment générer plusieurs `Request`
simultanément, correspondant par exemple, à différents liens sur la page
courante. Ils sont insérés séquentiellement dans le scheduler.

Pour cela on modifie la méthode `parse()` de façon à ce qu'elle retourne
un objet `Request` pour chaque nouveau lien rencontré. On associe
également à cet objet une fonction de callback qui déterminera la
manière dont cette nouvelle page doit être extraite.

Par exemple, pour que la spider continue dans les liens des différentes
régions (pour l'instant la fonction de callback ne fait rien) :

In [57]:
list(zip([1, 2, 3, 4], ["a", "b", "c", "d"]))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]

In [6]:
# %load newscrawler/newscrawler/spiders/lemonde_v2.py
import scrapy
from scrapy import Request


class LemondeSpider(scrapy.Spider):
    name = "lemondev2"
    allowed_domains = ["www.lemonde.fr"]
    start_urls = ['https://www.lemonde.fr']

    def parse(self, response):
        title = response.css('title::text').extract_first()
        all_links = {
            name:response.urljoin(url) for name, url in zip(
            response.css("#nav-markup .Nav__item")[3].css("a::text").extract(),
            response.css("#nav-markup .Nav__item")[3].css("a::attr(href)").extract())
        }
        yield {
            "title":title,
            "all_links":all_links
        }
        

Pour lancer la spider

In [7]:
!cd newscrawler && scrapy crawl lemondev2

2019-12-18 08:28:39 [scrapy.utils.log] INFO: Scrapy 1.5.1 started (bot: newscrawler)
2019-12-18 08:28:39 [scrapy.utils.log] INFO: Versions: lxml 4.3.2.0, libxml2 2.9.4, cssselect 1.0.3, parsel 1.5.0, w3lib 1.20.0, Twisted 18.9.0, Python 3.7.3 (default, Apr  3 2019, 05:39:12) - [GCC 8.3.0], pyOpenSSL 19.0.0 (OpenSSL 1.1.1c  28 May 2019), cryptography 2.6.1, Platform Linux-4.19.0-6-amd64-x86_64-with-debian-10.1
2019-12-18 08:28:39 [scrapy.crawler] INFO: Overridden settings: {'BOT_NAME': 'newscrawler', 'NEWSPIDER_MODULE': 'newscrawler.spiders', 'ROBOTSTXT_OBEY': True, 'SPIDER_MODULES': ['newscrawler.spiders']}
2019-12-18 08:28:39 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.memusage.MemoryUsage',
 'scrapy.extensions.logstats.LogStats']
2019-12-18 08:28:39 [scrapy.middleware] INFO: Enabled downloader middlewares:
['scrapy.downloadermiddlewares.robotstxt.RobotsTxtMiddleware',
 'scrapy.d

On veut ensuite *entrer* dans les liens des différentes sous-catégories
pour récupérer les articles. Pour cela, nous créons une méthode
`parse_category()` prend en argument un objet `Response` qui sera la
réponse correspondant aux liens des régions. On peut comme ceci
traverser un site en définissant des méthodes différentes en fonction du
type de contenu.

Si la structure du site est plus profonde, on peut empiler autant de
couches que souhaité.

Quand on arrive sur une page d'une sous-catégorie, on peut vouloir
récupérer tous les éléments de la page. Pour cela, on réutilise le
scrapy Shell pour commencer le développement de la nouvelle méthode
d'extraction.

Par exemple pour la page `https://www.lemonde.fr/international/` :

In [ ]:
scrapy shell 'https://www.lemonde.fr/international/'

Le fil des articles est stocké dans une balise avec la classe
`class=river`.

In [ ]:
In [3]: response.css(".river")
Out[3]:
[<Selector xpath="descendant-or-self::*[@class and contains(concat(' ', normalize-space(@class), ' '), ' fleuve ')]" data='<div class="fleuve">\n   <section>\n      '>,
<Selector xpath="descendant-or-self::*[@class and contains(concat(' ', normalize-space(@class), ' '), ' fleuve ')]" data='<div class="fleuve">\n</div>'>]

Pour récupérer chacun des articles, il faut adresser les balises
`<article>` contenues dans le sélecteur:

In [ ]:
In [4]: response.css(".river")[0].css(".teaser")
Out[4]:
[<Selector xpath='descendant-or-self::article' data='<article class="grid_12 alpha enrichi mg'>,
<Selector xpath='descendant-or-self::article' data='<article class="grid_12 alpha enrichi">\n'>,
<Selector xpath='descendant-or-self::article' data='<article class="grid_12 alpha enrichi">\n'>,
<Selector xpath='descendant-or-self::article' data='<article class="grid_12 alpha enrichi">\n'>,
<Selector xpath='descendant-or-self::article' data='<article class="grid_12 alpha enrichi">\n'>,
<Selector xpath='descendant-or-self::article' data='<article class="grid_12 alpha enrichi">\n'>,
<Selector xpath='descendant-or-self::article' data='<article class="grid_12 alpha enrichi">\n'>,
<Selector xpath='descendant-or-self::article' data='<article class="grid_12 alpha enrichi">\n'>]   

Comme précédemment, on peut empiler les sélecteurs `css` pour créer des
requêtes plus complexes.

Par exemple, pour récupérer tous les titres des différents articles :

In [ ]:
In [8]: response.css(".river")[0].css(".teaser h3::text").extract()
Out[8]:
['Des dizaines de milliers de Géorgiens contestent dans la rue l’élection de Salomé Zourabichvili\r\n\r\n\r\n',
'A Budapest en Hongrie, un îlot décroissant pour favoriser la transition\r\n\r\n\r\n',
'En Israël, la police recommande l’inculpation de Nétanyahou dans une troisième enquête\r\n\r\n\r\n',
'Donald Trump veut «\xa0mettre fin\xa0» à l’Aléna rapidement\r\n\r\n\r\n',
'Le cauchemar de la «\xa0rééducation\xa0» des musulmans en Chine\r\n\r\n',
'\r\n',
'«\xa0AMLO\xa0» lance sa transformation du Mexique\r\n\r\n\r\n',
'«\xa0Paris brûle\xa0»\xa0: les médias étrangers relatent le «\xa0chaos\xa0» en marge des défilés des «\xa0gilets jaunes\xa0»\r\n\r\n\r\n',
'Andrés Manuel Lopez Obrador intronisé président du Mexique\r\n\r\n\r\n']

En HTML les données sont souvent de très mauvaise qualité. Il faut
définir des méthodes permettant de les nettoyer pour être intégrées dans
des bases de données.

Par exemple, pour supprimer tous les espaces superflus :

In [8]:
def clean_spaces(string_):
    if string_ is not None: 
        return " ".join(string_.split())

Pour l'appliquer à tous les titres récupérés, on peut faire une list
comprehension : 

In [ ]:
In [11]: [clean_spaces(article) for article in response.css(".river")[0].css(".teaser h3::text").extract()]  

Out[11]: ['Des dizaines de milliers de Géorgiens contestent dans la rue l’élection de Salomé Zourabichvili',
          'A Budapest en Hongrie, un îlot décroissant pour favoriser la transition', 
          'En Israël, la police recommande l’inculpation de Nétanyahou dans une troisième enquête',
          'Donald Trump veut « mettre fin » à l’Aléna rapidement', 
          'Le cauchemar de la « rééducation » des musulmans en Chine',
          '',
          '« AMLO » lance sa transformation du Mexique', 
          '« Paris brûle » : les médias étrangers relatent le « chaos » en marge des défilés des « gilets jaunes »',
          'Andrés Manuel Lopez Obrador intronisé président du Mexique'
         ]


La méthode précédente est intéressante si l'on ne recherche qu'une seule
information par article.

Par contre si l'on veut récupérer d'autres caractéristiques comme
l'image ou la description par exemple, il est plus intéressant et plus
efficace de récupérer l'objet et d'effectuer plusieurs traitements sur
ce dernier.

Chaque objet retourné par les requêtes `css` est un selecteur avec
lequel on peut interagir.

Par exemple pour récupérer le titre et le prix

In [ ]:
# In [25]: for article in response.css(".fleuve")[0].css("article"):
# ...:     title = clean_spaces(article.css("h3 a::text").extract_first())
# ...:     image = article.css("img::attr(data-src)").extract_first()
# ...:     description = article.css(".txt3::text").extract_first()
# ...:     print(f"Title {title} \nImage {image}\nDescription {description}\n ----")


In [ ]:
In [25]: for article in response.css(".river")[0].css(".teaser"):
...:     title = clean_spaces(article.css("h3::text").extract_first())
...:     image = article.css("img::attr(data-src)").extract_first()
...:     description = article.css("p::text").extract_first()
...:     print(f"Title {title} \nImage {image}\nDescription {description}\n ----")


In [ ]:
Title Des dizaines de milliers de Géorgiens contestent dans la rue l’élection de Salomé Zourabichvili
Image https://s1.lemde.fr/image/2018/12/02/147x97/5391641_7_5874_les-partisans-de-l-opposant-grigol-vashadze_20d2e8693a49b83fd3c5578f7799ae9c.jpg
Description Elue présidente (un rôle essentiellement symbolique en Géorgie), l’ex-diplomate française, candidate du pouvoir, est contestée par l’opposition.
----
Title A Budapest en Hongrie, un îlot décroissant pour favoriser la transition
Image https://img.lemde.fr/2018/12/01/10/0/4214/2809/147/97/60/0/15b32ca_1EY4qISQ_BP4kPAh1fozJdXZ.jpg
Description Le centre logistique Cargonomia sert de matrice aux coopératives de l’économie durable et solidaire hongroise.
----
Title En Israël, la police recommande l’inculpation de Nétanyahou dans une troisième enquête
Image https://img.lemde.fr/2018/12/02/167/0/4207/2804/147/97/60/0/9e02c9b_3580d043ebc94b48b0f2cfef4e9a21e7-3580d043ebc94b48b0f2cfef4e9a21e7-0.jpg
Description Le premier ministre est soupçonné de corruption, fraude et abus de pouvoir, dans une affaire impliquant le groupe de télécoms israélien Bezeq.
----
Title Donald Trump veut « mettre fin » à l’Aléna rapidement
Image https://img.lemde.fr/2018/11/30/0/0/4861/3240/147/97/60/0/8b87184_5826023-01-06.jpg
Description Le président américain souhaite voir disparaître l’accord de libre-échange remontant à 1994 avec le Mexique et le Canada, qu’il qualifie régulièrement de « pire accord jamais signé », en faveur du nouveau traité négocié difficilement avec ses voisins nord-américains ces derniers mois.
----
Title Le cauchemar de la « rééducation » des musulmans en Chine
Image https://img.lemde.fr/2018/11/15/151/0/5000/3333/147/97/60/0/118c78f_248b226e6b91450aa8a68bd0ea5525a8-248b226e6b91450aa8a68bd0ea5525a8-0.jpg
Description Ouïgours et Kazakhs du Xinjiang... C’est toute une population musulmane que Pékin veut « rééduquer » en internant des centaines de milliers d’entre eux dans des camps.
----
Title « AMLO » lance sa transformation du Mexique
Image https://img.lemde.fr/2018/12/02/45/0/1497/998/147/97/60/0/a33c174_GGGTBR84_MEXICO-POLITICS-_1202_11.JPG
Description Education et santé gratuites, hausse du salaire minimum, bourses scolaires : à peine investi, le président Andres Manuel Lopez Obrador a listé les mesures qu’il entend prendre pour redresser le pays.
----
Title « Paris brûle » : les médias étrangers relatent le « chaos » en marge des défilés des « gilets jaunes »
Image https://img.lemde.fr/2018/12/02/361/0/598/396/147/97/60/0/ba46a6e_XVIt1Ffwm50iYBheccVieUQQ.jpg
Description Les images de destructions, d’échauffourées ou de voitures enflammées s’affichaient samedi soir en « une » de nombreux sites d’actualité internationaux.
----
Title Andrés Manuel Lopez Obrador intronisé président du Mexique
Image https://img.lemde.fr/2018/12/02/91/145/1346/897/147/97/60/0/877cd51_a4618baa8da2414bb62bab28a6d4c745-a4618baa8da2414bb62bab28a6d4c745-0.jpg
Description Le nouveau chef d’Etat a promis de lutter contre la corruption en menant une transformation « profonde et radicale » du pays.
----

## Persistence des données

Pour pouvoir stocker les informations que l'on récupère en parcourant un
site il faut pouvoir les stocker. On utilise soit de simples
dictionnaires Python, ou mieux des `scrapy.Item` qui sont des
dictionnaires améliorés.

Nous allons voir les deux façons de faire. On peut réécrire la méthode
`parse_category()` pour lui faire retourner un dictionnaire
correspondant à chaque offre rencontrée.

In [9]:
def parse_category(self, response):
    for article in response.css(".river")[0].css(".teaser"):
        title = self.clean_spaces(article.css("h3 ::text").extract_first())
        image = article.css("img::attr(data-src)").extract_first()
        description = article.css("p::text").extract_first()
        yield {
            "title":title,
            "image":image,
            "description":description
        }

Si on combine tout dans la spider :

In [10]:
# %load newscrawler/newscrawler/spiders/lemonde_v3.py
import scrapy
from scrapy import Request


class LemondeSpider(scrapy.Spider):
    name = "lemondev3"
    allowed_domains = ["www.lemonde.fr"]
    start_urls = ['https://www.lemonde.fr']

    def parse(self, response):
        title = response.css('title::text').extract_first()
        all_links = {
            name:response.urljoin(url) for name, url in zip(
            response.css("#nav-markup .Nav__item")[3].css("a::text").extract(),
            response.css("#nav-markup .Nav__item")[3].css("a::attr(href)").extract())
        }
        for link in all_links.values():
            yield Request(link, callback=self.parse_category)
            
    def parse_category(self, response):
        for article in response.css(".river")[0].css(".teaser"):
            title = self.clean_spaces(article.css("h3::text").extract_first())
            image = article.css("img::attr(data-src)").extract_first()
            description = article.css("p::text").extract_first()
            yield {
                "title":title,
                "image":image,
                "description":description
            }

    def clean_spaces(self, string):
        if string:
            return " ".join(string.split())

On peut maintenant lancer notre spider avec la commande suivante :

In [ ]:
scrapy crawl <NAME>

`scrapy crawl` permet de démarrer le processus en allant chercher la
classe `scrapy.Spider` dont l'attribut `name` = &lt;NAME&gt;.

Par exemple, pour la spider `LeMondeSpider` :

In [11]:
!cd newscrawler && scrapy crawl lemondev3

2019-12-18 09:08:22 [scrapy.utils.log] INFO: Scrapy 1.5.1 started (bot: newscrawler)
2019-12-18 09:08:22 [scrapy.utils.log] INFO: Versions: lxml 4.3.2.0, libxml2 2.9.4, cssselect 1.0.3, parsel 1.5.0, w3lib 1.20.0, Twisted 18.9.0, Python 3.7.3 (default, Apr  3 2019, 05:39:12) - [GCC 8.3.0], pyOpenSSL 19.0.0 (OpenSSL 1.1.1c  28 May 2019), cryptography 2.6.1, Platform Linux-4.19.0-6-amd64-x86_64-with-debian-10.1
2019-12-18 09:08:22 [scrapy.crawler] INFO: Overridden settings: {'BOT_NAME': 'newscrawler', 'NEWSPIDER_MODULE': 'newscrawler.spiders', 'ROBOTSTXT_OBEY': True, 'SPIDER_MODULES': ['newscrawler.spiders']}
2019-12-18 09:08:22 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.memusage.MemoryUsage',
 'scrapy.extensions.logstats.LogStats']
2019-12-18 09:08:22 [scrapy.middleware] INFO: Enabled downloader middlewares:
['scrapy.downloadermiddlewares.robotstxt.RobotsTxtMiddleware',
 'scrapy.d

2019-12-18 09:08:23 [scrapy.core.engine] DEBUG: Crawled (200) <GET https://www.lemonde.fr/international/> (referer: https://www.lemonde.fr)
2019-12-18 09:08:23 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/actualite-medias/>
{'title': '« La littérature jeunesse est-elle de la “vraie” littérature ? »', 'image': 'https://img.lemde.fr/2019/11/04/0/0/5150/3433/110/74/60/0/01105fb_rpQ8GB5XJ1X-jNuT65_BheBn.jpg', 'description': 'Alors que le Salon du livre et de la presse jeunesse de Montreuil a ouvert ses portes le 27\xa0novembre, Guillaume Nail, président de la Charte des auteurs et illustrateurs jeunesse, s’insurge, dans une tribune au «\xa0Monde\xa0», contre le statut secondaire trop souvent attribué à cette littérature.'}
2019-12-18 09:08:23 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/actualite-medias/>
{'title': 'Vidéo à la demande, publicité, quota de chanson française : le CSA veut améliorer la loi audiovisuelle', 'image': 'https://img

2019-12-18 09:08:23 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/sante/>
{'title': 'Dans la Nièvre, 2 000 personnes manifestent pour sauver la clinique de Cosne-Cours-sur-Loire', 'image': 'https://img.lemde.fr/2019/11/23/0/37/703/469/110/74/60/0/926481d_YF6OGva4vGQd6b2mrVubNHDg.PNG', 'description': 'La clinique du Nohain a été placée en redressement judiciaire. Son propriétaire, le groupe Kapa Santé, est impliqué dans une enquête pour abus de biens sociaux.'}
2019-12-18 09:08:23 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/sport/>
{'title': 'Ligue des champions : et si les vainqueurs de groupe choisissaient leur adversaire pour les huitièmes de finale ?', 'image': 'https://img.lemde.fr/2019/12/16/0/0/4764/3176/110/74/60/0/3246bf9_5230014-01-06.jpg', 'description': 'Mathématicien et amateur de football, Julien Guyon propose de remplacer le tirage au sort par un exercice de choix stratégiques qui, selon lui, «\xa0redonnerait de l’intérêt 

2019-12-18 09:08:23 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/podcasts/>
{'title': '« Vous ne voulez quand même pas qu’on aille à l’hôtel ? »', 'image': 'https://img.lemde.fr/2019/12/10/0/0/1999/1333/110/74/60/0/bdbd030_wKnQ6n8L39D3ebmlKpMltjp7.png', 'description': 'Retrouvez le témoignage de Christian, 70\xa0ans, interprété par Jonathan Zaccaï. Un podcast issu de notre série «\xa0S’aimer comme on se quitte\xa0».'}
2019-12-18 09:08:23 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/podcasts/>
{'title': 'Lomepal : « La folie de ma grand-mère, Jeannine, me fascinait »', 'image': 'https://img.lemde.fr/2018/02/15/203/0/3248/2157/110/74/60/0/b536b26_3096-a7j6tp.j232.jpg', 'description': 'Qu’est-ce qu’avoir du goût\xa0? Qui a bon ou mauvais goût\xa0? Comment se forme-t-il\xa0? Est-il le produit d’une éducation, d’un milieu social ou au contraire le fruit d’une construction personnelle\xa0? Cette semaine, le chanteur français répond aux questi

2019-12-18 09:08:24 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/sport/>
{'title': 'Noël Le Graët pense que les clubs de football « ont relâché la pression » sur leurs supporteurs', 'image': 'https://img.lemde.fr/2019/12/15/29/0/4000/2666/110/74/60/0/266af9c_jeGdy6y-4VIKhAb_otoHpUFf.jpg', 'description': 'Le président de la fédération française en appelle à la responsabilité des dirigeants de clubs après une série d’incidents dans ou aux abords des stades.'}
2019-12-18 09:08:24 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/sport/>
{'title': 'Revivez la seconde étape de la Coupe du monde de biathlon : cinq Français dans le top 10', 'image': 'https://img.lemde.fr/2019/12/13/0/0/4464/2976/110/74/60/0/7064150_d2a29c1780794a73b87f879d7b35f4c0-d2a29c1780794a73b87f879d7b35f4c0-0.jpg', 'description': 'Le quintuple champion olympique Martin Fourcade a fini seulement à la 10e place. Son compatriote Emilien Jacquelin s’installe pour la première fois

...

2019-12-18 09:08:24 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/societe/>
{'title': '« Ça hurle et ça klaxonne. Les gens deviennent des bêtes ! » : plongée dans le chaos urbain d’un Paris éprouvé par la grève', 'image': 'https://img.lemde.fr/2019/12/09/2/0/5000/3333/110/74/60/0/476e33d_uK1m9e7_VuKIk-avbtpAbqeA.JPG', 'description': 'La mobilisation contre la réforme des retraites, lancée le 5 décembre, met à l’épreuve les nerfs des automobilistes et des usagers des transports en Ile-de-France.'}
2019-12-18 09:08:24 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/societe/>
{'title': 'Charles Gardou, spécialiste du handicap : « Il n’y a pas de société inclusive sans remise en cause de prés carrés »', 'image': 'https://img.lemde.fr/2019/12/13/0/0/5137/3425/110/74/60/0/f60cbed_tt1ovDxmPwaTlD79U9kyG7Uj.jpg', 'description': 'Charles Gardou, anthropologue et spécialiste du handicap, analyse le rapport de notre culture aux personnes handicapées, n

2019-12-18 09:08:24 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/pixels/>
{'title': 'Les preuves de l’ingérence russe dans la campagne de Macron en 2017', 'image': 'https://img.lemde.fr/2019/12/07/94/0/3388/2258/110/74/60/0/6bd65ff_NCKBbLJCi9VqmVGgd8vT4eCc.jpg', 'description': 'Pour la première fois, des éléments recueillis par « Le Monde » accréditent l’implication de deux groupes de hackeurs liés au renseignement russe.'}
2019-12-18 09:08:24 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/pixels/>
{'title': 'Votre enfant joue à « Fortnite » et vous n’en pouvez plus, racontez-nous', 'image': 'https://img.lemde.fr/2019/12/06/12/0/5200/3466/110/74/60/0/4e5436d_34tNaWtu5SOqNlzxQCpPRZtU.jpg', 'description': 'Cris, excitation, parties à rallonge, vocabulaire incompréhensible… votre enfant se transforme lorsqu’il joue.'}
2019-12-18 09:08:24 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/pixels/>
{'title': 'Les 6 meilleurs

2019-12-18 09:08:24 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/donald-trump/>
{'title': '« Vous voyez, nous pouvons parvenir à un accord ensemble » : Echange de prisonniers entre l’Iran et les Etats-Unis', 'image': 'https://img.lemde.fr/2019/12/07/85/0/2048/1365/110/74/60/0/3cb3796_DBA01_IRAN-USA-PRISONERS_1207_11.JPG', 'description': 'Donald Trump s’est réjoui d’une opération négociée par l’intermédiaire de la Suisse.'}
2019-12-18 09:08:24 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/donald-trump/>
{'title': 'Dennis Ross : « Les Etats-Unis ont perdu leur crédibilité au Proche-Orient »', 'image': 'https://img.lemde.fr/2019/12/05/0/0/9449/6299/110/74/60/0/d8418b8_qUfZhYe9roDGUJRg3C8WPPRS.jpg', 'description': 'L’expert américain du Proche-Orient estime, dans un entretien au «\xa0Monde\xa0», que, en amorçant puis en consolidant un retrait de la région, les présidents Obama et Trump se sont privés de leviers pour agir sur les crises en co

2019-12-18 09:08:24 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/debat-sur-les-retraites/>
{'title': 'Réforme des retraites : les syndicats enseignants maintiennent la pression après des réunions bilatérales', 'image': 'https://img.lemde.fr/2019/12/13/0/0/3417/2278/110/74/60/0/f6b75e4_5217928-01-06.jpg', 'description': 'Le 13\xa0décembre, Jean-Michel Blanquer a ouvert des négociations qui devraient se poursuivre jusqu’en juin\xa02020. Les syndicats, reçus lors d’une série de rencontres bilatérales, maintiennent leur appel à la grève pour le 17\xa0décembre.'}
2019-12-18 09:08:24 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/debat-sur-les-retraites/>
{'title': 'En soutien de la contestation, le Parti socialiste durcit sa position sur les retraites', 'image': 'https://img.lemde.fr/2019/12/16/0/0/5899/3933/110/74/60/0/a2aeddb_uR_p-zWUuNe4L-1cprd8a9Hx.jpg', 'description': 'Le PS, qui participe aux manifestations contre la réforme des retraite

2019-12-18 09:08:24 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/mobilisation-du-5-decembre/>
{'title': 'Les féministes s’affichent ensemble pour dire non à une réforme des retraites « sexiste »', 'image': 'https://img.lemde.fr/2019/12/17/0/0/5749/3833/110/74/60/0/db28718_RGaxwTXGmXUYY-iMbpYfcfS9.jpg', 'description': 'Trois cents personnes ont assisté lundi à Paris, à un meeting sur le thème «\xa0Femmes et retraites\xa0», organisé à l’initiative de la députée de Seine-Saint-Denis, Clémentine Autain.'}
2019-12-18 09:08:24 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/mobilisation-du-5-decembre/>
{'title': 'Grève du 17 décembre : le trafic SNCF et RATP restera fortement perturbé mardi', 'image': 'https://img.lemde.fr/2019/12/15/0/8/3483/2322/110/74/60/0/183915a_GGGPAR02_FRANCE-PROTESTS-PENSIONS_1215_11.JPG', 'description': 'La SNCF prévoit un TGV sur quatre et un Transilien sur cinq, tandis que huit lignes de métro resteront fermées à Pari

On peut exporter les résultats de ces retours dans différents formats de
fichiers.

-   CSV : `scrapy crawl lemonde -o lbc.csv`
-   JSON : `scrapy crawl lemonde -o lbc.json`
-   JSONLINE : `scrapy crawl lemonde -o lbc.jl`
-   XML : `scrapy crawl lemonde -o lbc.xml`

### Exercice :

Exécuter la spider avec les différents formats de stockage.
Explorer ensuite le contenu des fichiers ainsi créés.

In [17]:
#!cd newscrawler && scrapy crawl lemondev3 -o lbc.csv
# !cd newscrawler && scrapy crawl lemondev3 -o lbc.json
# !cd newscrawler && scrapy crawl lemondev3 -o lbc.jl
!cd newscrawler && scrapy crawl lemondev3 -o lbc.xml

2019-12-18 09:17:17 [scrapy.utils.log] INFO: Scrapy 1.5.1 started (bot: newscrawler)
2019-12-18 09:17:17 [scrapy.utils.log] INFO: Versions: lxml 4.3.2.0, libxml2 2.9.4, cssselect 1.0.3, parsel 1.5.0, w3lib 1.20.0, Twisted 18.9.0, Python 3.7.3 (default, Apr  3 2019, 05:39:12) - [GCC 8.3.0], pyOpenSSL 19.0.0 (OpenSSL 1.1.1c  28 May 2019), cryptography 2.6.1, Platform Linux-4.19.0-6-amd64-x86_64-with-debian-10.1
2019-12-18 09:17:17 [scrapy.crawler] INFO: Overridden settings: {'BOT_NAME': 'newscrawler', 'FEED_FORMAT': 'xml', 'FEED_URI': 'lbc.xml', 'NEWSPIDER_MODULE': 'newscrawler.spiders', 'ROBOTSTXT_OBEY': True, 'SPIDER_MODULES': ['newscrawler.spiders']}
2019-12-18 09:17:17 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.memusage.MemoryUsage',
 'scrapy.extensions.feedexport.FeedExporter',
 'scrapy.extensions.logstats.LogStats']
2019-12-18 09:17:17 [scrapy.middleware] INFO: Enabled downlo

2019-12-18 09:17:18 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/big-browser/>
{'title': 'D’Auschwitz 1943 à New York 2016, une histoire d’amour contrariée par l’Histoire', 'image': 'https://img.lemde.fr/2019/12/06/0/0/5256/3504/110/74/60/0/f8cfe13_a707ab545ab24a5d98e360b4635f50bd-a707ab545ab24a5d98e360b4635f50bd-0.jpg', 'description': 'Le «\xa0New York Times\xa0» raconte l’histoire de David Wisnia et Helen Spitzer, qui sont tombés amoureux à Auschwitz avant d’être séparés à la libération des camps. Ils ne se sont retrouvés que 72\xa0ans plus tard.'}
2019-12-18 09:17:18 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/big-browser/>
{'title': '« Le violeur, c’est toi », une chanson chilienne contre les violences sexistes fait le tour du monde', 'image': 'https://img.lemde.fr/2019/11/30/47/0/3637/2424/110/74/60/0/3b8fd80_5133031-01-06.jpg', 'description': 'La performance du collectif chilien féministe Las Tesis pour dénoncer les violences gen

2019-12-18 09:17:18 [scrapy.core.engine] DEBUG: Crawled (200) <GET https://www.lemonde.fr/les-decodeurs/> (referer: https://www.lemonde.fr)
2019-12-18 09:17:18 [scrapy.core.engine] DEBUG: Crawled (200) <GET https://www.lemonde.fr/actualite-en-continu/> (referer: https://www.lemonde.fr)
2019-12-18 09:17:18 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/disparitions/>
{'title': 'Bertrand Landrieu, ancien directeur du cabinet du président Chirac, est mort', 'image': 'https://img.lemde.fr/2019/12/16/0/0/921/614/110/74/60/0/395049c_Eo2obOo4NHAjdt0rn3dWTqUn.jpg', 'description': 'Ce haut fonctionnaire a fait la connaissance de l’ancien chef de l’Etat en\xa01973 et a dirigé son cabinet à l’Elysée entre 1995 et 2002. Fidèle parmi les fidèles, il restera auprès de Jacques Chirac après son départ du pouvoir, en\xa02007. Il est mort le 6\xa0décembre, à l’âge de 74\xa0ans.'}
2019-12-18 09:17:18 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/disparitions

2019-12-18 09:17:18 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/sante/>
{'title': 'Du vin quotidien à la tolérance zéro', 'image': 'https://img.lemde.fr/2019/12/17/0/0/5703/3802/110/74/60/0/e64e4ac_xAlomHs1LzHrJlMYhvz8FMDB.jpg', 'description': 'Dans sa chronique, Anne Rodier rappelle les dangers d’une consommation d’alcool importante sur les lieux de travail. Sans en appeler à une abstinence générale pour 2020 elle insiste sur une plus grande vigilance sur ce fléau.'}
2019-12-18 09:17:18 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/actualite-medias/>
{'title': 'Les 75 ans du « Monde » : vingt-quatre heures dans la rédaction', 'image': 'https://img.lemde.fr/2019/12/16/0/0/4000/2666/110/74/60/0/b31b378_kW11uLS7Zq7gjlYRxGTd5jqT.jpg', 'description': 'Le quotidien et son site, plus indissociables que jamais, obéissent à une mécanique très particulière. Un exemple avec la journée du vendredi 6\xa0décembre.'}
2019-12-18 09:17:18 [scrapy.core.

2019-12-18 09:17:18 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/actualite-medias/>
{'title': 'Droits télé : M6 et TF1 vont diffuser l’Euro 2020 de football', 'image': 'https://img.lemde.fr/2019/11/27/0/0/4276/2851/110/74/60/0/064dab9_rRYeX1WktX6i8nU3JamGmbnJ.jpg', 'description': 'Les deux chaînes privées vont diffuser en clair «\xa0les 23\xa0meilleures affiches\xa0» de la compétition, qui se déroulera dans douze villes européennes.'}
2019-12-18 09:17:18 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/actualite-medias/>
{'title': 'Un conseil de déontologie journalistique sera créé en décembre', 'image': 'https://img.lemde.fr/2019/11/25/0/0/5476/3651/110/74/60/0/060a90b_jPrWDV-k3TuNP6fViWl37PvJ.jpg', 'description': 'La création d’une telle instance fait débat depuis des années. Un de ses objectifs est notamment de répondre à l’énorme défiance envers les médias.'}
2019-12-18 09:17:18 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www

...

2019-12-18 09:17:19 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/les-decodeurs/>
{'title': 'Le système de retraites a-t-il 150 milliards d’euros en réserve ?', 'image': 'https://img.lemde.fr/2019/12/06/2/0/5000/3333/110/74/60/0/92b5c03_-wVWrMjq9GSrSTBDtX6mrZCL.JPG', 'description': 'Un économiste affirme qu’il existe une manne destinée aux retraites, pour dénoncer l’«\xa0enfumage\xa0» de la réforme. Vrai ou faux\xa0?'}
2019-12-18 09:17:19 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/les-decodeurs/>
{'title': 'A l’Assemblée nationale, la majorité LRM continue de s’éroder', 'image': 'https://img.lemde.fr/2019/12/05/0/27/573/382/110/74/60/0/4098b33_iw3HPhXOlO-gvMtAA9WG3hsJ.png', 'description': 'EN\xa0UN GRAPHIQUE. A mi-chemin de la législature, le groupe La République en marche s’est réduit de dix sièges, et il n’est pas exclu qu’il en perde encore dans l’avenir.'}
2019-12-18 09:17:19 [scrapy.core.scraper] DEBUG: Scraped from <200 https://w

2019-12-18 09:17:19 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/debat-sur-les-retraites/>
{'title': 'Le député Laurent Pietraszewski nommé secrétaire d’Etat chargé des retraites pour remplacer Jean-Paul Delevoye', 'image': 'https://img.lemde.fr/2019/12/18/0/0/5217/3478/110/74/60/0/dc3ede5_wRWTyESOTPCc1yAGz69zKKFT.jpg', 'description': 'L’élu La République en marche du Nord aura la difficile tâche de reprendre les négociations avec les syndicats après treize jours de grèves dans les transports.'}
2019-12-18 09:17:19 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/debat-sur-les-retraites/>
{'title': '« Sous le sapin, la grève » : dans les manifestations contre la réforme des retraites, la « trêve de Noël » est largement rejetée', 'image': 'https://img.lemde.fr/2019/12/18/0/0/6000/4000/110/74/60/0/2c4e005_Ppo_Ho5aXY6vd7_cOk_EjFta.jpg', 'description': 'Les opposants au projet du gouvernement n’ont pas désarmé, mardi 17\xa0décembre. Nombre d’en

2019-12-18 09:17:19 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/climat/>
{'title': 'Malgré son retard, l’Espagne affiche son ambition climatique', 'image': 'https://img.lemde.fr/2019/12/03/0/0/5760/3840/110/74/60/0/c1f8347_yb7XyVsqgm_SfI8Yv2Uw1Slj.jpg', 'description': 'Le gouvernement socialiste, qui se veut en pointe dans la lutte contre le réchauffement en accueillant la COP25, place l’accompagnement social au premier plan.'}
2019-12-18 09:17:19 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/climat/>
{'title': 'A la COP climat de Madrid, Antonio Guterres appelle à « ne pas trahir la famille humaine »', 'image': 'https://img.lemde.fr/2019/12/03/0/0/5472/3648/110/74/60/0/a40f91a_5148764-01-06.jpg', 'description': 'Plusieurs chefs d’Etat présents lundi 2\xa0décembre ont tenté de répondre à l’inquiétude des jeunes générations.'}
2019-12-18 09:17:19 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/climat/>
{'title': 'Cl

2019-12-18 09:17:19 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/referendum-sur-le-brexit/>
{'title': 'Boris Johnson tétanisé à l’idée d’un dérapage de Donald Trump à Londres', 'image': 'https://img.lemde.fr/2019/12/03/27/642/2115/1410/110/74/60/0/8d25d52_GDN80010_NATO-SUMMIT-DOWNING-STREET_1203_11.JPG', 'description': 'En visite pour les 70\xa0ans de l’OTAN, le président américain a assuré qu’il n’avait «\xa0aucune intention de s’immiscer\xa0» dans la campagne politique britannique.'}
2019-12-18 09:17:19 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/referendum-sur-le-brexit/>
{'title': 'Royaume-Uni : ces bastions Labour tentés par Boris Johnson', 'image': 'https://img.lemde.fr/2019/12/02/0/10/4849/3233/110/74/60/0/f56672d_QDiZVvRhpsfnrG16cYEytn5U.jpg', 'description': 'Le premier ministre conservateur veut ravir aux travaillistes, lors des législatives du 12 décembre, ses fiefs d’Angleterre du Nord, favorables à la sortie de l’UE.'}
2019

2019-12-18 09:17:19 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/debat-sur-les-retraites/>
{'title': 'Les policiers vont garder leur régime dérogatoire de retraites', 'image': 'https://img.lemde.fr/2019/12/13/0/1/3498/2332/110/74/60/0/8444d34_CHP10_FRANCE-SECURITY-_1213_11.JPG', 'description': 'Les principaux syndicats ont obtenu gain de cause, le ministère de l’intérieur justifiant la décision par le fait que «\xa0la fonction même de policier les expose au risque\xa0».'}
2019-12-18 09:17:19 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/debat-sur-les-retraites/>
{'title': 'Réforme des retraites : l’âge pivot est « négociable », répète le gouvernement', 'image': 'https://img.lemde.fr/2019/12/11/2/0/5000/3333/110/74/60/0/e6afb68_OjUprYYxCvNvN1VL9pF1Kxci.JPG', 'description': 'La secrétaire d’Etat auprès du ministre de l’économie, Agnès Pannier-Runacher, est revenue sur l’une des mesures les plus contestées par les opposants à la réforme, no


## Votre premier Item

La classe `Item` permet de structurer les données que l'on souhaite
récupérer sous la forme d'un modèle. Les items doivent être définis dans
le fichier `items.py` créé par la commande `scrapy startproject`. Les
`Item` héritent de la class `scrapy.Item`.

On veut structurer les données avec deux champs : le titre et le prix de
l'annonce. Scrapy utilise une classe `scrapy.Field` permettant de
'déclarer' ces champs. Dans notre cas :

In [18]:
# %load newscrawler/newscrawler/items.py

# Define here the models for your scraped items
#
# See documentation in:
# https://docs.scrapy.org/en/latest/topics/items.html

import scrapy

class ArticleItem(scrapy.Item):
    title = scrapy.Field()
    image = scrapy.Field()
    description = scrapy.Field()

Utiliser la classe `scrapy.Item` plutôt qu'un simple dictionnaire permet
plus de contrôle sur la structure des données. En effet, on ne peut
insérer dans les items que des données avec des clés 'déclarées'. Ce qui
assure une plus grande cohérence au sein d'un projet.

On peut instancier un item de plusieurs façons :

In [19]:
article_item = ArticleItem(title="Gilets Jaunes", image=None, description="Un samedi de manifestations")

In [20]:
print(article_item)

{'description': 'Un samedi de manifestations',
 'image': None,
 'title': 'Gilets Jaunes'}


In [21]:
article_item = ArticleItem()
article_item["title"] = 'Gilets Jaunes'
article_item["description"] = 'Un samedi de manifestations'

In [22]:
print(article_item)

{'description': 'Un samedi de manifestations', 'title': 'Gilets Jaunes'}


La définition d'un item permet de palier toutes les erreurs de typo dans
les champs.

In [27]:
article_item = ArticleItem()
article_item["titelkjwnxvmnscbvmknxc"] = 'Gilets Jaunes'

KeyError: 'ArticleItem does not support field: titelkjwnxvmnscbvmknxc'

Les items héritent des dictionnaires Python, et possèdent donc toutes
les méthodes de ceux-ci:

In [23]:
article_item = ArticleItem(title="Gilets Jaunes")
print(article_item["title"]) # Méthode __getitem__()
print(article_item.get("description", "no description provided")) # Méthode get()

Gilets Jaunes
no description provided


On peut transformer un `Item` en dictionnaire très facilement, en le
passant au constructeur:

In [24]:
article_item = ArticleItem(title="Drone DJI")
print(type(article_item))
dict_item = dict(article_item)
print(type(dict_item))
print(dict_item)

<class '__main__.ArticleItem'>
<class 'dict'>
{'title': 'Drone DJI'}


On intègre maintenant cet item dans notre spider.

In [28]:
!ls

citations_churchill_spider1.py	data	Introduction.ipynb  __pycache__
citations_churchill_spider2.py	images	newscrawler	    result.json


In [31]:
# -*- coding: utf-8 -*-
import scrapy
from scrapy import Request
from items import ArticleItem
class LemondeSpider(scrapy.Spider):
    name = "lemondev4"
    allowed_domains = ["www.lemonde.fr"]
    start_urls = ['https://www.lemonde.fr']

    def parse(self, response):
        title = response.css('title::text').extract_first()
        all_links = {
            name:response.urljoin(url) for name, url in zip(
            response.css("#nav-markup .Nav__item")[3].css("a::text").extract(),
            response.css("#nav-markup .Nav__item")[3].css("a::attr(href)").extract())
        }
        for link in all_links.values():
            yield Request(link, callback=self.parse_category)

    def parse_category(self, response):
        for article in response.css(".river")[0].css(".teaser"):
            title = self.clean_spaces(article.css("h3 ::text").extract_first())
            image = article.css("img::attr(data-src)").extract_first()
            description = article.css("p::text").extract_first()

            yield ArticleItem(
                title=title,
                image=image,
                description=description
            )

    def clean_spaces(self, string):
        if string:
            return " ".join(string.split())

On voit bien que le générateur retourne maintenant un `Item`.

### Exercice : 

Relancer la spider pour vérifier le bon déroulement de l'extraction.


In [35]:
!cd newscrawler/ && scrapy crawl lemondev4

2019-12-18 09:38:53 [scrapy.utils.log] INFO: Scrapy 1.5.1 started (bot: newscrawler)
2019-12-18 09:38:53 [scrapy.utils.log] INFO: Versions: lxml 4.3.2.0, libxml2 2.9.4, cssselect 1.0.3, parsel 1.5.0, w3lib 1.20.0, Twisted 18.9.0, Python 3.7.3 (default, Apr  3 2019, 05:39:12) - [GCC 8.3.0], pyOpenSSL 19.0.0 (OpenSSL 1.1.1c  28 May 2019), cryptography 2.6.1, Platform Linux-4.19.0-6-amd64-x86_64-with-debian-10.1
2019-12-18 09:38:53 [scrapy.crawler] INFO: Overridden settings: {'BOT_NAME': 'newscrawler', 'NEWSPIDER_MODULE': 'newscrawler.spiders', 'ROBOTSTXT_OBEY': True, 'SPIDER_MODULES': ['newscrawler.spiders']}
2019-12-18 09:38:53 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.memusage.MemoryUsage',
 'scrapy.extensions.logstats.LogStats']
2019-12-18 09:38:53 [scrapy.middleware] INFO: Enabled downloader middlewares:
['scrapy.downloadermiddlewares.robotstxt.RobotsTxtMiddleware',
 'scrapy.d

2019-12-18 09:38:54 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/campus/>
{'description': 'Alors que de plus en plus de lycéens se tournent vers la '
                'langue de Shakespeare, les établissements anglophones se '
                'multiplient dans les grandes villes du royaume.',
 'image': 'https://img.lemde.fr/2019/11/20/0/0/3499/2333/110/74/60/0/735dae0_QLlm0OtGOq6P1zKcH_W2Rgh3.jpg',
 'title': 'Au Maroc, l’anglais s’affirme face à l’enseignement en français'}
2019-12-18 09:38:54 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/campus/>
{'description': 'Le rendez-vous traditionnel est discuté dans les '
                'établissements, car la réforme en cours transforme '
                'l’équilibre des classes et le calendrier de l’année.',
 'image': 'https://img.lemde.fr/2019/11/21/2/0/6000/4000/110/74/60/0/d892390_av2W5vHgI5iF020zaSCSNBF4.jpg',
 'title': 'Désuet pour les uns, crucial pour les autres : le conseil de classe '

2019-12-18 09:38:54 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/disparitions/>
{'description': 'Mathématicien de formation, l’ingénieur a conçu et développé '
                'les véhicules à bas coûts pour la marque au losange fabriqués '
                'en Roumanie, en Inde puis en Chine. Il est mort le 5\xa0'
                'décembre, à l’âge de 73\xa0ans.',
 'image': 'https://img.lemde.fr/2019/12/16/31/0/2103/1402/110/74/60/0/8953cb5_JojJIP4ESNjdk1kb7MAsLDYF.jpg',
 'title': 'La mort de Gérard Detourbet, l’inventeur de la gamme « low cost » '
          'de Renault'}
2019-12-18 09:38:54 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/afrique/>
{'description': 'Le mouvement de contestation, le\xa0Hirak, mobilise depuis '
                'dix mois la jeunesse et les forces vives de la société civile '
                'contre un pouvoir militaire autocratique décidé à durer par '
                'tous les moyens, dont un simulacre d’élec

2019-12-18 09:38:54 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/afrique/>
{'description': 'Notre chroniqueuse Guillemette Faure s’est rendue au siège de '
                'l’OCDE pour rencontrer des pros de l’éducation et autres '
                'invités plus ou moins concernés.',
 'image': 'https://img.lemde.fr/2019/10/03/0/0/2362/1574/110/74/60/0/556d773_gZO8uXgRUbkyqARUpcgRrI0d.jpg',
 'title': 'Avec les experts de l’enseignement à la présentation du rapport '
          'PISA'}
2019-12-18 09:38:54 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/afrique/>
{'description': 'L’ancien président est toujours sous le coup d’un mandat '
                'd’arrêt international pour «\xa0crimes contre l’humanité et '
                'incitation au génocide\xa0».',
 'image': 'https://img.lemde.fr/2019/12/16/0/0/2200/1466/110/74/60/0/1084c4c_btz36qi5grW-awcO2s6hgPW-.jpg',
 'title': 'Centrafrique : retour à Bangui de François Bozizé après six ans '


2019-12-18 09:38:54 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/disparitions/>
{'description': 'Directeur musical du prestigieux Concertgebouw d’Amsterdam et '
                'de l’Orchestre symphonique de la Radio bavaroise, le chef '
                's’est éteint à Saint-Pétersbourg, à l’âge de 76\xa0ans.',
 'image': 'https://img.lemde.fr/2019/12/02/0/15/2995/1997/110/74/60/0/064e27d_8tP7G_PTX1uv9Fpm8MBi5dRd.jpg',
 'title': 'La mort du chef d’orchestre letton Mariss Jansons'}
2019-12-18 09:38:54 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/disparitions/>
{'description': 'Au cours de ses cinq années au pouvoir (1982-1987), le '
                'patriarche du parti conservateur a été la cheville ouvrière '
                'de l’ouverture de l’Archipel sur le monde. Il est mort le 29 '
                'novembre, à l’âge de 101 ans.',
 'image': 'https://img.lemde.fr/2019/11/29/0/0/5315/3543/110/74/60/0/d5635bf__WamJXlbWthECpxrEStQDUqh.j

...

2019-12-18 09:38:54 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/climat/>
{'description': 'Des champs d’Afrique à certaines fermes franciliennes, des '
                'agriculteurs ont recours à des insectes pour lutter contre '
                'd’autres insectes ravageurs pour leurs cultures. Une '
                'alternative aux pesticides aussi efficace que spectaculaire.',
 'image': 'https://img.lemde.fr/2019/11/28/0/140/1519/1013/110/74/60/0/d1fab39_A5oW68bK96XIi0ou7l6objSO.jpg',
 'title': 'Plan B : comment des insectes peuvent sauver des millions de vies '
          'humaines'}
2019-12-18 09:38:54 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/climat/>
{'description': 'Valérie Masson-Delmotte, paléoclimatologue, regrette '
                'l’absence d’accord sur les mécanismes du marché carbone lors '
                'de la COP25, qui s’est achevée dimanche à Madrid.',
 'image': 'https://img.lemde.fr/2019/12/16/10/0/3922/2614/110/

2019-12-18 09:38:55 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/referendum-sur-le-brexit/>
{'description': 'Dans une tribune au «\xa0Monde\xa0», le juriste Sébastien '
                'Platon énumère la liste des embûches politiques, juridiques '
                'et économiques qui peuvent conduire à un «\xa0no deal\xa0» '
                'après les élections remportées par les conservateurs au '
                'Royaume-Uni.',
 'image': 'https://img.lemde.fr/2019/12/16/265/0/4218/2805/110/74/60/0/57e0d5e_xFvrLiPovMboWfo5d9snzvTK.jpg',
 'title': 'Brexit : « L’accord de retrait ne règle pas de façon pérenne les '
          'relations entre l’UE et le Royaume-Uni »'}
2019-12-18 09:38:55 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/referendum-sur-le-brexit/>
{'description': 'Sorti largement victorieux du scrutin, Boris Johnson a joué '
                'vendredi l’apaisement après des mois de discours polarisant, '
                'tendant

2019-12-18 09:38:55 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/debat-sur-les-retraites/>
{'description': 'La mobilisation contre la réforme des retraites, lancée le 5 '
                'décembre, met à l’épreuve les nerfs des automobilistes et des '
                'usagers des transports en Ile-de-France.',
 'image': 'https://img.lemde.fr/2019/12/09/2/0/5000/3333/110/74/60/0/476e33d_uK1m9e7_VuKIk-avbtpAbqeA.JPG',
 'title': '« Ça hurle et ça klaxonne. Les gens deviennent des bêtes ! » : '
          'plongée dans le chaos urbain d’un Paris éprouvé par la grève'}
2019-12-18 09:38:55 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/debat-sur-les-retraites/>
{'description': 'La Haute Autorité pour la transparence de la vie publique '
                'doit décider mercredi si elle saisit la justice concernant '
                'les multiples «\xa0oublis\xa0» de M. Delevoye dans sa '
                'déclaration d’intérêts.',
 'image': 'https:/

2019-12-18 09:38:55 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/mobilisation-du-5-decembre/>
{'description': 'Dix jours après le début du mouvement, le soutien des '
                'Français à la grève est stable. Chaque camp reste ferme, avec '
                'en perspective le blocage des transports pendant les fêtes.',
 'image': 'https://img.lemde.fr/2019/12/11/4/0/5900/3933/110/74/60/0/4e2bc73_eNRFtyYaAJzIuSBmhp6yZmL8.jpg',
 'title': 'Réforme des retraites : entre le gouvernement et les syndicats, la '
          'bataille de l’opinion'}
2019-12-18 09:38:55 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/mobilisation-du-5-decembre/>
{'description': 'Depuis vendredi, le gouvernement met la pression sur les '
                'grévistes pour faire cesser la mobilisation, notamment à la '
                'SNCF, contre la réforme des retraites durant les fêtes.',
 'image': 'https://img.lemde.fr/2018/06/01/0/0/6240/4160/110/74/60/0/98e708a

2019-12-18 09:38:55 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/mobilisation-du-5-decembre/>
{'description': 'Le premier ministre a dévoilé mercredi le détail de la '
                'réforme des retraites. Si le Medef salue «\xa0un bon '
                'équilibre\xa0», Philippe Martinez (CGT) estime que «\xa0le '
                'gouvernement s’est moqué du monde\xa0».',
 'image': 'https://img.lemde.fr/2019/12/11/0/0/3600/2400/110/74/60/0/adac93c_yRDOB8wL7k5ciRNj8As28ELy.jpg',
 'title': 'Réforme des retraites : retrouvez les annonces d’Edouard Philippe '
          'et les réactions'}
2019-12-18 09:38:55 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/mobilisation-du-5-decembre/>
{'description': 'A la veille des annonces d’Edouard Philippe, les députés, de '
                'LR au PCF ont fait subir au gouvernement un houleux '
                'réquisitoire.',
 'image': 'https://img.lemde.fr/2019/12/10/0/2/3972/2648/110/74/60/0/349f49a_5

## Postprocessing

Si l'on se réfère au diagramme d'architecture de Scrapy, on voit qu'il
est possible d'insérer des composants supplémentaires dans le flux de
traitement. Ces composants s'appellent `Pipelines`.

Par défaut, tous les `Item` générés au sein d'un projet Scrapy passent
par les `Pipelines`. Les pipelines sont utilisés la plupart du temps
pour :

-   Nettoyer du contenu HTML ;
-   Valider les données scrapées ;
-   Supprimer les items qu'on ne souhaite pas stocker ;
-   Stocker ces objets dans des bases de données.

Les pipelines doivent être définis dans le fichier `pipelines.py`.

Dans notre cas on peut vouloir nettoyer le champ `title` pour enlever
les caractères superflus.

Nous allons alors transferer la fonction de nettoyage du code html dans
une Pipeline.

In [36]:
# %load newscrawler/newscrawler/pipelines.py

# Define your item pipelines here
#
# Don't forget to add your pipeline to the ITEM_PIPELINES setting
# See: https://docs.scrapy.org/en/latest/topics/item-pipeline.html

from scrapy.exceptions import DropItem

class TextPipeline(object):

    def process_item(self, item, spider):
        if item['title']:
            item["title"] = clean_spaces(item["title"])
            return item
        else:
            raise DropItem("Missing title in %s" % item)


def clean_spaces(string):
    if string:
        return " ".join(string.split())

Pour dire au process Scrapy de faire transiter les items par ces
pipelines. Il faut le spécifier dans le fichier de paramétrage
`settings.py`.

In [37]:
ITEM_PIPELINES = {
     'newscrawler.pipelines.TextPipeline': 300,
 }

On peut maintenant supprimer la fonction `clean_spaces()` de
l'extraction des données et laisser le Pipeline faire son travail. La
valeur entière définie permet de déterminer l'ordre dans lequel les
pipelines vont être appelés. Ces entiers peuvent être compris entre 0 et
1000.

On relance notre spider :

In [45]:
!cd newscrawler && scrapy crawl lemondev4 -o ../data/articles.json

2019-12-18 09:58:46 [scrapy.utils.log] INFO: Scrapy 1.5.1 started (bot: newscrawler)
2019-12-18 09:58:46 [scrapy.utils.log] INFO: Versions: lxml 4.3.2.0, libxml2 2.9.4, cssselect 1.0.3, parsel 1.5.0, w3lib 1.20.0, Twisted 18.9.0, Python 3.7.3 (default, Apr  3 2019, 05:39:12) - [GCC 8.3.0], pyOpenSSL 19.0.0 (OpenSSL 1.1.1c  28 May 2019), cryptography 2.6.1, Platform Linux-4.19.0-6-amd64-x86_64-with-debian-10.1
2019-12-18 09:58:46 [scrapy.crawler] INFO: Overridden settings: {'BOT_NAME': 'newscrawler', 'FEED_FORMAT': 'json', 'FEED_URI': '../data/articles.json', 'NEWSPIDER_MODULE': 'newscrawler.spiders', 'ROBOTSTXT_OBEY': True, 'SPIDER_MODULES': ['newscrawler.spiders']}
2019-12-18 09:58:46 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.memusage.MemoryUsage',
 'scrapy.extensions.feedexport.FeedExporter',
 'scrapy.extensions.logstats.LogStats']
2019-12-18 09:58:46 [scrapy.middleware] INFO:

2019-12-18 09:58:47 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/actualite-medias/>
{'description': 'Ces sites basés en Russie et en Ukraine ciblent à la fois les '
                'soutiens de Marine Le\xa0Pen, de Jean-Luc Mélenchon et des '
                '«\xa0gilets jaunes\xa0».',
 'image': 'https://img.lemde.fr/2019/11/21/0/0/1284/856/110/74/60/0/f904f81_KmQbfDikVqC8iqoaD01rQEK1.png',
 'title': 'Derrière de pseudo-sites politiques français, un vaste réseau venu '
          'd’Ukraine'}
2019-12-18 09:58:47 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/actualite-medias/>
{'description': 'Les reporters d’«\xa0Iwacu\xa0» sont accusés de «\xa0'
                'complicité d’atteinte à la sécurité de l’Etat\xa0». Le pays '
                'est à la 159e place du classement mondial de la liberté de la '
                'presse de RSF.',
 'image': 'https://img.lemde.fr/2019/11/22/0/55/703/469/110/74/60/0/db38abd_n3bX_ISptjh-TGeMMxTnz4n3.JP

2019-12-18 09:58:47 [scrapy.core.engine] DEBUG: Crawled (200) <GET https://www.lemonde.fr/sante/> (referer: https://www.lemonde.fr)
2019-12-18 09:58:47 [scrapy.core.engine] DEBUG: Crawled (200) <GET https://www.lemonde.fr/big-browser/> (referer: https://www.lemonde.fr)
2019-12-18 09:58:47 [scrapy.core.engine] DEBUG: Crawled (200) <GET https://www.lemonde.fr/disparitions/> (referer: https://www.lemonde.fr)
2019-12-18 09:58:47 [scrapy.core.engine] DEBUG: Crawled (200) <GET https://www.lemonde.fr/afrique/> (referer: https://www.lemonde.fr)
2019-12-18 09:58:47 [scrapy.core.engine] DEBUG: Crawled (200) <GET https://www.lemonde.fr/sport/> (referer: https://www.lemonde.fr)
2019-12-18 09:58:47 [scrapy.core.engine] DEBUG: Crawled (200) <GET https://www.lemonde.fr/les-decodeurs/> (referer: https://www.lemonde.fr)
2019-12-18 09:58:48 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/sante/>
{'description': 'Dans sa chronique, Anne Rodier rappelle les dangers d’une '
          

2019-12-18 09:58:48 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/big-browser/>
{'description': 'Le «\xa0New York Times\xa0» raconte l’histoire de David '
                'Wisnia et Helen Spitzer, qui sont tombés amoureux à Auschwitz '
                'avant d’être séparés à la libération des camps. Ils ne se '
                'sont retrouvés que 72\xa0ans plus tard.',
 'image': 'https://img.lemde.fr/2019/12/06/0/0/5256/3504/110/74/60/0/f8cfe13_a707ab545ab24a5d98e360b4635f50bd-a707ab545ab24a5d98e360b4635f50bd-0.jpg',
 'title': 'D’Auschwitz 1943 à New York 2016, une histoire d’amour contrariée '
          'par l’Histoire'}
2019-12-18 09:58:48 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/big-browser/>
{'description': 'La performance du collectif chilien féministe Las Tesis pour '
                'dénoncer les violences genrées a été reproduite à travers la '
                'planète.',
 'image': 'https://img.lemde.fr/2019/11/30/47/0/3637/2

2019-12-18 09:58:48 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/sante/>
{'description': 'Les pompiers professionnels, comme ceux que «\xa0Le Monde\xa0'
                '» a pu suivre en intervention à Rennes, passent le plus clair '
                'de leur temps à faire du «\xa0social\xa0».',
 'image': 'https://img.lemde.fr/2019/11/29/167/0/4000/2666/110/74/60/0/99c7f1a_-vmx3nd8-j86bjJPxZx1lcIW.jpg',
 'title': '« Nous sommes devenus les médecins des pauvres » : les pompiers '
          'face à l’évolution du métier'}
2019-12-18 09:58:48 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/sante/>
{'description': 'Pour Sandrine Fournier, directrice du service des programmes '
                'France de Sidaction, l’une des clés de la sensibilisation des '
                '15-24\xa0ans réside dans la mise en place de programmes mieux '
                'adaptés à l’école.',
 'image': 'https://img.lemde.fr/2019/11/22/0/0/2868/1912/110/74/60/0/cc9

...

2019-12-18 09:58:49 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/campus/>
{'description': 'Quand notre cerveau se développe-t-il et comment intègre-t-il '
                'les informations ?',
 'image': 'https://img.lemde.fr/2019/11/19/0/248/3309/2206/110/74/60/0/1107127_uBv8u7ZNZ8NZq8RKyYcc7D5q.jpg',
 'title': '« Neuromythes » et réalités sur notre cerveau'}
2019-12-18 09:58:49 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/campus/>
{'description': 'Le sociologue et médecin Nicholas Christakis estime que si '
                'l’intelligence artificielle peut être bénéfique, il faut '
                'doter les machines de principes fondamentaux afin qu’elles ne '
                'nuisent pas aux sociétés humaines.',
 'image': 'https://img.lemde.fr/2019/11/15/0/0/1200/800/110/74/60/0/deaabe8_WJARZU5yMw3EQKsZbgF6B5h3.jpg',
 'title': '« L’idée de confier les apprentissages à une machine est '
          'inquiétante, voire inhumaine »'}
2019

2019-12-18 09:58:49 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/mobilisation-du-5-decembre/>
{'description': 'Sous la menace de nouveaux blocages dans les transports '
                'pendant les fêtes, le premier ministre tente, mercredi, de '
                'trouver une porte de sortie à la crise.',
 'image': 'https://img.lemde.fr/2019/12/18/2/0/5000/3333/110/74/60/0/e9bc20f_aLpvya90hssDlOnk4hmJVEeS.JPG',
 'title': 'Réforme des retraites : au 14e jour de grève, Edouard Philippe '
          'reçoit à nouveau les syndicats à Matignon'}
2019-12-18 09:58:49 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/debat-sur-les-retraites/>
{'description': 'Sous la menace de nouveaux blocages dans les transports '
                'pendant les fêtes, le premier ministre tente, mercredi, de '
                'trouver une porte de sortie à la crise.',
 'image': 'https://img.lemde.fr/2019/12/18/2/0/5000/3333/110/74/60/0/e9bc20f_aLpvya90hssDlOnk4hmJVEeS.

2019-12-18 09:58:49 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/algerie/>
{'description': 'Au cœur de la capitale, des milliers de personnes brandissent '
                'mercredi des cartons rouges en signe de refus de la tenue de '
                'ce scrutin.',
 'image': 'https://img.lemde.fr/2019/12/11/0/4/4920/3280/110/74/60/0/10c88ca_0m4SbHKDgSGHR_FQVYld9Bit.jpg',
 'title': 'En Algérie, manifestation anti-élection à Alger à 24 heures de la '
          'présidentielle'}
2019-12-18 09:58:49 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/algerie/>
{'description': 'Hautement symbolique en raison de leur histoire commune, la '
                'seule rencontre entre les deux pays a eu lieu en octobre\xa0'
                '2001.',
 'image': 'https://img.lemde.fr/2019/12/11/54/0/4077/2718/110/74/60/0/4a76068_pAR3ApbhI890i2rSyyHnNFtS.jpg',
 'title': 'Le match France-Algérie n’aura probablement pas lieu en 2020'}
2019-12-18 09:58:49 [scrapy

2019-12-18 09:58:49 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/algerie/>
{'description': 'Le scrutin est prévu jeudi. Il est très critiqué par le '
                'mouvement populaire de contestation Hirak, pour qui il '
                'permettra surtout au «\xa0système\xa0» de se régénérer.',
 'image': 'https://img.lemde.fr/2019/12/06/0/20/2424/1616/110/74/60/0/d8d83cb_yWns2wR80vcCrOJ7nIRWT7g7.jpg',
 'title': '« Eh Gaïd Salah, oublie le vote ! » : manifestation massive à Alger '
          'avant l’élections présidentielle'}
2019-12-18 09:58:49 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/algerie/>
{'description': 'Accusés de dilapidation de deniers publics, Abdelmalek Sellal '
                'et Ahmed Ouyahia comparaissent aux côtés d’ex-ministres, de '
                'hauts fonctionnaires et d’hommes d’affaires.',
 'image': 'https://img.lemde.fr/2019/12/06/0/28/2277/1518/110/74/60/0/8c8ceef_WNXKCLZQ10J3Wy3RUq4QCegh.jpg',
 'title

2019-12-18 09:58:49 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/referendum-sur-le-brexit/>
{'description': 'Pour la sortie de l’UE, le gouvernement britannique avait '
                'prévu la fabrication de millions de pièces commémoratives de '
                '50\xa0pence... qui seront broyées et fondues.',
 'image': 'https://img.lemde.fr/2019/10/31/0/1/3478/2319/110/74/60/0/8cf3443_XZV3c5n6N_vDgG98OKoGPIca.jpg',
 'title': 'Le Brexit repoussé, les pièces de monnaie commémoratives seront '
          'détruites'}
2019-12-18 09:58:49 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.lemonde.fr/referendum-sur-le-brexit/>
{'description': 'Au Royaume-Uni, avec l’arrivée comme premier ministre de '
                'Boris Johnson, les travaux des économistes ou des think tanks '
                'tant conservateurs que progressistes s’attachent désormais à '
                'explorer les politiques économiques alternatives qui '
                'pourraient

On peut aussi utiliser les Pipelines pour stocker les données récupérées
dans une base de données. Pour stocker les items dans des documents
mongo :

In [46]:
import pymongo

class MongoPipeline(object):

    collection_name = 'scrapy_items'

    def open_spider(self, spider):
        self.client = pymongo.MongoClient()
        self.db = self.client["lemonde"]

    def close_spider(self, spider):
        self.client.close()

    def process_item(self, item, spider):
        self.db[self.collection_name].insert_one(dict(item))
        return item

Ici on redéfinit deux autres méthodes: `open_spider()`et
`close_spider()`, ces méthodes sont appelées comme leur nom l'indique,
lorsque la Spider est instanciée et fermée.

Ces méthodes nous permettent d'ouvrir la connexion Mongo et de la fermer
lorsque le scraping se termine. La méthode `process_item()` est appelé à
chaque fois qu'un item passe dans le mécanisme interne de scrapy. Ici,
la méthode permet d'insérer l'item en tant que document mongo.

Pour que ce pipeline soit appelé il faut l'ajouter dans les settings du
projet.

In [ ]:
ITEM_PIPELINES = {
    'newscrawler.pipelines.TextPipeline': 100,
    'newscrawler.pipelines.MongoPipeline': 300
}

Le pipeline est ajoutée à la fin du process pour profiter des deux
précédents.

## Settings

Scrapy permet de gérer le comportement des spiders avec certains
paramètres. Comme expliqué dans le premier cours, il est important de
suivre des règles en respectant la structure des différents sites. Il
existe énormément de paramètres mais nous allons (dans le cadre de ce
cours) aborder les plus utilisés :

-   DOWNLOAD\_DELAY : Le temps de téléchargement entre chaque requête
    sur le même domaine ;
-   CONCURRENT\_REQUESTS\_PER\_DOMAIN : Nombre de requêtes simultanées
    par domaine ;
-   CONCURRENT\_REQUESTS\_PER\_IP : Nombre de requêtes simultanées par
    IP ;
-   DEFAULT\_REQUEST\_HEADERS : Headers HTTP utilisés pour les requêtes
    ;
-   ROBOTSTXT\_OBEY : Scrapy récupère le robots.txt et adapte le
    scraping en fonction des règles trouvées ;
-   USER\_AGENT : UserAgent utilisé pour faire les requêtes ;
-   BOT\_NAME : Nom du bot annoncé lors des requêtes
-   HTTPCACHE\_ENABLED : Utilisation du cache HTTP, utile lors du
    parcours multiple de la même page.

Le fichiers `settings.py` permet de définir les paramètres globaux d'un
projet. Si votre projet contient un grand nombre de spiders, il peut
être intéressant d'avoir des paramètres distincts pour chaque spider. Un
moyen simple est d'ajouter un attribut `custom_settings` à votre spider
:

In [ ]:
class LeMondeSpider(scrapy.Spider):
        name = "lemonde"
        allowed_domains = ["lemonde.fr"]
        start_urls = ['http://lemonde.fr/']
        custom_settings = {
            "HTTPCACHE_ENABLED":True, 
            "CONCURRENT_REQUESTS_PER_DOMAIN":100
        }